# FloodWatch – Mapping flood damage from space (Multimodal AI Hackathon 2026, Track B)
Educational prototype, not an operational tool.

**Attribution (required):**
Contains modified Copernicus Sentinel data 2026.
Produced using Copernicus WorldDEM-30 © DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018 provided under COPERNICUS by the European Union and ESA; all rights reserved.
© OpenStreetMap contributors. (ODbL)
No model is trained, so no training dataset is used (Kuro Siwo and Sen1Floods11 are NOT used).

**Pipeline:** Sentinel-1 (same-track before/after) + Sentinel-2 -> flood & debris mask -> OSM damage
-> road graph cut-off analysis -> DEM flood-path trace (bonus) -> LLM situation-report copilot (EN/NE)
-> folium dashboard + one-page HTML report.

## 1. Install

In [1]:
!pip -q install rasterio geopandas shapely pyproj networkx scipy folium requests matplotlib

## 2. Configuration & secrets
Add these in Colab -> left sidebar key icon ("Secrets"), notebook access ON:
`CDSE_ID`, `CDSE_SECRET` (Copernicus Data Space OAuth client) and `GROQ_API_KEY` (free key from console.groq.com).
**Never paste keys into a cell** - this file is meant to be uploaded to GitHub.

In [2]:
import os, io, re, json, math, time, getpass, base64, datetime as dt
import numpy as np
import requests
import rasterio
from rasterio import features
from rasterio.io import MemoryFile
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
import geopandas as gpd
import shapely
from shapely.geometry import shape, LineString
import networkx as nx
from pyproj import Transformer
import folium
import matplotlib
import matplotlib.pyplot as plt

CONFIG = dict(
    # Area of interest (west, south, east, north) in lon/lat. Example: Bhote Koshi-Trishuli corridor (approximate, adjust it!)
    bbox=(85.10, 27.90, 85.45, 28.32),
    flood_date="2026-08-26",          # day of the event
    osm_date="2026-07-27",            # MUST be before the flood date (ohsome snapshot)
    res=30,                           # metres per pixel (auto-coarsened if the area is too big)
    upstream_point=(85.38, 28.28),    # (lon, lat) for the flood-path bonus; set None to skip
    out_dir="outputs",
    # thresholds
    water_vv_db=-18.0, water_vh_db=-24.0, change_db=3.0,
    max_slope_deg=25.0, max_hand_m=60.0, min_blob_px=8,
    use_optical=True,
)
os.makedirs(CONFIG["out_dir"], exist_ok=True)
assert CONFIG["osm_date"] < CONFIG["flood_date"], "OSM snapshot must be BEFORE the event (data rules)."

def get_secret(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    v = os.environ.get(name)
    if not v and required:
        v = getpass.getpass(f"{name}: ")
    return v

# Keys come from Colab Secrets / environment variables. Nothing secret is stored in this notebook.
CDSE_ID = get_secret("CDSE_ID"); CDSE_SECRET = get_secret("CDSE_SECRET")
GROQ_KEY = get_secret("GROQ_API_KEY")
ANTHROPIC_KEY = GROQ_KEY   # legacy name: ask() treats the copilot as enabled when a key is set


CDSE_ID: ··········
CDSE_SECRET: ··········
GROQ_API_KEY: ··········


## 3. Copernicus Data Space (Sentinel Hub Process API) helpers
One API gives us orthorectified, terrain-corrected Sentinel-1, Sentinel-2 L2A and Copernicus DEM,
already cropped and resampled to a common grid.

In [3]:
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
PROCESS_URL = "https://sh.dataspace.copernicus.eu/api/v1/process"
CATALOG_URL = "https://sh.dataspace.copernicus.eu/api/v1/catalog/1.0.0/search"
OHSOME_URL = "https://api.ohsome.org/v1/elements/geometry"

_tok = {"v": None, "t": 0.0}
def auth_headers():
    if not _tok["v"] or time.time() - _tok["t"] > 420:
        r = requests.post(TOKEN_URL, data=dict(grant_type="client_credentials",
                          client_id=CDSE_ID, client_secret=CDSE_SECRET), timeout=60)
        r.raise_for_status()
        _tok.update(v=r.json()["access_token"], t=time.time())
    return {"Authorization": f"Bearer {_tok['v']}"}

class Grid:
    """Common metric (UTM) grid used by every raster."""
    def __init__(self, bbox, res):
        w, s, e, n = bbox
        lon0, lat0 = (w + e) / 2, (s + n) / 2
        self.epsg = (32600 if lat0 >= 0 else 32700) + int((lon0 + 180) // 6) + 1
        tr = Transformer.from_crs(4326, self.epsg, always_xy=True)
        pts = [tr.transform(x, y) for x, y in [(w, s), (w, n), (e, s), (e, n)]]
        minx, maxx = min(p[0] for p in pts), max(p[0] for p in pts)
        miny, maxy = min(p[1] for p in pts), max(p[1] for p in pts)
        while max((maxx - minx), (maxy - miny)) / res > 2400:
            res *= 1.25                                   # Process API limit is 2500 px
        self.res = res
        self.W = int(math.ceil((maxx - minx) / res)); self.H = int(math.ceil((maxy - miny) / res))
        self.bounds = (minx, miny, minx + self.W * res, miny + self.H * res)
        self.transform = rasterio.transform.from_origin(minx, miny + self.H * res, res, res)
        self.to_ll = Transformer.from_crs(self.epsg, 4326, always_xy=True)
        self.from_ll = tr

def process(data, evalscript, grid):
    body = {"input": {"bounds": {"bbox": list(grid.bounds),
                                 "properties": {"crs": f"http://www.opengis.net/def/crs/EPSG/0/{grid.epsg}"}},
                      "data": data},
            "output": {"width": grid.W, "height": grid.H,
                       "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
            "evalscript": evalscript}
    r = requests.post(PROCESS_URL, json=body, headers=auth_headers(), timeout=600)
    if r.status_code != 200:
        raise RuntimeError(f"Process API {r.status_code}: {r.text[:400]}")
    with MemoryFile(r.content) as mf, mf.open() as ds:
        return ds.read().astype("float32")

EVAL_DEM = """//VERSION=3
function setup(){return{input:["DEM"],output:{bands:1,sampleType:"FLOAT32"}}}
function evaluatePixel(s){return[s.DEM]}"""
EVAL_S1 = """//VERSION=3
function setup(){return{input:["VV","VH","dataMask"],output:{bands:3,sampleType:"FLOAT32"}}}
function evaluatePixel(s){return[s.VV,s.VH,s.dataMask]}"""
EVAL_S2 = """//VERSION=3
function setup(){return{input:["B03","B04","B08","B11","SCL","dataMask"],output:{bands:6,sampleType:"FLOAT32"}}}
function evaluatePixel(s){return[s.B03,s.B04,s.B08,s.B11,s.SCL,s.dataMask]}"""

def get_dem(grid):
    d = [{"type": "dem", "dataFilter": {"demInstance": "COPERNICUS_30"}}]
    dem = process(d, EVAL_DEM, grid)[0]
    dem[~np.isfinite(dem)] = np.nanmedian(dem)
    return dem

def s1_scene(grid, day, orbit):
    flt = {"timeRange": {"from": f"{day}T00:00:00Z", "to": f"{day}T23:59:59Z"},
           "acquisitionMode": "IW", "polarization": "DV", "resolution": "HIGH"}
    if orbit: flt["orbitDirection"] = orbit
    d = [{"type": "sentinel-1-grd", "dataFilter": flt,
          "processing": {"orthorectify": True, "demInstance": "COPERNICUS_30",
                         "backCoeff": "GAMMA0_TERRAIN", "speckleFilter": {"type": "NONE"}}}]
    return process(d, EVAL_S1, grid)

def s2_composite(grid, t0, t1):
    d = [{"type": "sentinel-2-l2a", "dataFilter": {
          "timeRange": {"from": f"{t0}T00:00:00Z", "to": f"{t1}T23:59:59Z"},
          "maxCloudCoverage": 80, "mosaickingOrder": "leastCC"}}]
    return process(d, EVAL_S2, grid)

def find_s1_pair(bbox, flood_day):
    """Find a before/after pair on the SAME orbit track (same direction, same time of day, multiple of 12 days apart)."""
    fd = dt.date.fromisoformat(flood_day)
    body = {"collections": ["sentinel-1-grd"], "bbox": list(bbox), "limit": 100,
            "datetime": f"{fd - dt.timedelta(days=48)}T00:00:00Z/{fd + dt.timedelta(days=24)}T23:59:59Z"}
    feats = []
    while True:
        r = requests.post(CATALOG_URL, json=body, headers=auth_headers(), timeout=120)
        r.raise_for_status(); j = r.json(); feats += j.get("features", [])
        nxt = j.get("context", {}).get("next")
        if not nxt: break
        body["next"] = nxt
    rows = []
    for f in feats:
        p = f["properties"]
        t = dt.datetime.fromisoformat(p["datetime"].replace("Z", "+00:00"))
        if p.get("sar:instrument_mode", "IW") != "IW": continue
        rows.append(dict(day=t.date(), tod=t.hour + t.minute / 60,
                         orbit=(p.get("sat:orbit_state") or "").upper() or None,
                         rel=p.get("sat:relative_orbit")))
    best = None
    for p in rows:
        if p["day"] <= fd: continue
        for q in rows:
            if q["day"] >= fd: continue
            dd = (p["day"] - q["day"]).days
            if (q["orbit"] == p["orbit"] and abs(q["tod"] - p["tod"]) < 0.3 and dd % 12 == 0
                    and (q["rel"] is None or p["rel"] is None or q["rel"] == p["rel"])):
                key = ((p["day"] - fd).days, (fd - q["day"]).days)
                if best is None or key < best[0]:
                    best = (key, q, p)
    if best is None:
        avail = sorted({(r["day"].isoformat(), r["orbit"], round(r["tod"], 1)) for r in rows})
        raise RuntimeError(f"No same-track S1 pair found. Available scenes (day, orbit, hour): {avail}")
    _, q, p = best
    return dict(pre_day=q["day"].isoformat(), post_day=p["day"].isoformat(), orbit=p["orbit"], rel=p["rel"])

## 4. Q1 – Where did the flood hit? (Sentinel-1 change detection + Sentinel-2 optical + terrain constraints)

In [4]:
def terrain(dem, res):
    gy, gx = np.gradient(ndi.gaussian_filter(dem, 1), res)
    return np.degrees(np.arctan(np.hypot(gx, gy)))

def hand_proxy(dem, seeds):
    """Height above the nearest permanent-water pixel (rough HAND)."""
    if seeds.sum() < 20:
        seeds = dem <= np.percentile(dem, 2)
    idx = ndi.distance_transform_edt(~seeds, return_distances=False, return_indices=True)
    return dem - dem[idx[0], idx[1]]

def s1_db(arr):
    """3-band (VV, VH, mask) -> speckle-reduced dB (NaN where invalid). Filter is done in linear power."""
    out = []
    valid = (arr[2] > 0) & (arr[0] > 0)
    for b in (arr[0], arr[1]):
        x = np.where(valid, b, 0.0)
        num = ndi.uniform_filter(x, 5); den = ndi.uniform_filter(valid.astype("float32"), 5)
        lin = np.where(den > 0.3, num / np.maximum(den, 1e-6), np.nan)
        out.append(10 * np.log10(np.clip(lin, 1e-5, None)))
    return out[0], out[1]

def clean(mask, min_px):
    mask = ndi.binary_opening(mask, structure=np.ones((3, 3)))
    lab, n = ndi.label(mask)
    if n == 0: return mask
    size = np.bincount(lab.ravel()); keep = size >= min_px; keep[0] = False
    return ndi.binary_fill_holes(keep[lab])

def detect_s1(pre, post, slope, hand, C):
    vv0, vh0 = s1_db(pre); vv1, vh1 = s1_db(post)
    ok = np.isfinite(vv0) & np.isfinite(vv1)
    perm_water = (vv0 < C["water_vv_db"]) & (slope < 5)
    water_pre = vv0 < C["water_vv_db"]
    shadow = vv0 < -22                                   # radar shadow / very dark before -> ignore
    zone = (slope < C["max_slope_deg"]) & (hand < C["max_hand_m"]) & ok
    new_water = (vv1 < C["water_vv_db"]) & (vh1 < C["water_vh_db"]) & ~water_pre & zone
    d = vv1 - vv0
    debris = (np.abs(d) > C["change_db"]) & ~new_water & ~shadow & zone
    return clean(new_water, C["min_blob_px"]), clean(debris, C["min_blob_px"]), perm_water, ok

def detect_s2(pre, post, slope, hand, C):
    def ndi_(a, b): return (a - b) / (a + b + 1e-6)
    def clear(x): return (x[5] > 0) & ~np.isin(x[4], [0, 1, 3, 8, 9, 10, 11])
    valid = clear(pre) & clear(post)
    frac = float(valid.mean())
    zone = valid & (slope < C["max_slope_deg"]) & (hand < C["max_hand_m"])
    mndwi0, mndwi1 = ndi_(pre[0], pre[3]), ndi_(post[0], post[3])
    ndvi0, ndvi1 = ndi_(pre[2], pre[1]), ndi_(post[2], post[1])
    water = (mndwi1 > 0.1) & (mndwi0 < 0.0) & zone
    debris = ((ndvi0 - ndvi1) > 0.3) & (ndvi1 < 0.25) & ~water & zone
    return clean(water, C["min_blob_px"]), clean(debris, C["min_blob_px"]), frac

def map_flood(grid, C):
    dem = get_dem(grid); slope = terrain(dem, grid.res)
    pair = find_s1_pair(C["bbox"], C["flood_date"])
    print("S1 pair:", pair)
    pre = s1_scene(grid, pair["pre_day"], pair["orbit"]); post = s1_scene(grid, pair["post_day"], pair["orbit"])
    vv0, _ = s1_db(pre)
    seeds = (vv0 < C["water_vv_db"]) & (slope < 5)
    seeds = ndi.binary_opening(seeds, np.ones((3, 3)))
    hand = hand_proxy(dem, seeds)
    w1, d1, perm, ok = detect_s1(pre, post, slope, hand, C)
    info = dict(s1_pre=pair["pre_day"], s1_post=pair["post_day"], orbit=pair["orbit"],
                s1_valid_pct=round(100 * float(ok.mean()), 1), s2_used=False, s2_clear_pct=0.0)
    w2 = d2 = np.zeros_like(w1)
    if C["use_optical"]:
        try:
            fd = dt.date.fromisoformat(C["flood_date"])
            s2pre = s2_composite(grid, fd - dt.timedelta(days=45), fd - dt.timedelta(days=1))
            s2post = s2_composite(grid, fd + dt.timedelta(days=1), fd + dt.timedelta(days=20))
            w2, d2, frac = detect_s2(s2pre, s2post, slope, hand, C)
            info.update(s2_clear_pct=round(100 * frac, 1), s2_used=frac > 0.15)
            if frac <= 0.15: w2 = d2 = np.zeros_like(w1)
        except Exception as e:
            print("Optical step skipped:", e)
    water = w1 | w2
    debris = (d1 | d2) & ~water
    cls = np.zeros(slope.shape, "uint8"); cls[debris] = 2; cls[water] = 1
    return dict(cls=cls, dem=dem, slope=slope, hand=hand, perm_water=perm, info=info)

def vectorize(cls, grid):
    rows = []
    for v, name in [(1, "water"), (2, "debris")]:
        m = cls == v
        for geom, _ in features.shapes(m.astype("uint8"), mask=m, transform=grid.transform):
            g = shape(geom).simplify(grid.res / 2)
            if not g.is_empty: rows.append(dict(kind=name, geometry=g))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=grid.epsg) if rows else \
           gpd.GeoDataFrame({"kind": []}, geometry=[], crs=grid.epsg)

## 5. Q2 – What was damaged? (pre-event OpenStreetMap via ohsome)

In [5]:
def ohsome(bbox, flt, time_, cols):
    r = requests.post(OHSOME_URL, data={"bboxes": ",".join(map(str, bbox)), "time": time_,
                                        "filter": flt, "properties": "tags"}, timeout=900)
    r.raise_for_status()
    fc = r.json().get("features", [])
    gdf = gpd.GeoDataFrame.from_features(fc, crs=4326) if fc else gpd.GeoDataFrame(geometry=[], crs=4326)
    for c in cols:
        if c not in gdf: gdf[c] = None
    return gdf

ROAD_CLASSES = "motorway,trunk,primary,secondary,tertiary,unclassified,residential,living_street,track," \
               "motorway_link,trunk_link,primary_link,secondary_link,tertiary_link"

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    osm = dict(
        buildings=ohsome(bb, "building=* and geometry:polygon", t, ["building", "name"]),
        roads=ohsome(bb, f"highway in ({ROAD_CLASSES}) and geometry:line", t, ["highway", "bridge", "name"]),
        places=ohsome(bb, "place in (city,town,village,hamlet) and geometry:point", t, ["place", "name"]),
        hospitals=ohsome(bb, "(amenity=hospital or healthcare=hospital) and (geometry:point or geometry:polygon)", t, ["name"]),
    )
    if osm["hospitals"].empty:
        print("No hospitals in AOI -> falling back to clinics")
        osm["hospitals"] = ohsome(bb, "amenity=clinic and (geometry:point or geometry:polygon)", t, ["name"])
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}

def damage_assessment(osm, flood, buffer_m=30):
    fz = shapely.union_all(flood.geometry.values) if len(flood) else shapely.Polygon()
    near = fz.buffer(buffer_m)
    b, r = osm["buildings"].copy(), osm["roads"].copy()
    b["hit"] = b.geometry.intersects(fz).values if len(b) else []
    b["at_risk"] = (~b["hit"]) & b.geometry.intersects(near).values if len(b) else []
    r["hit"] = r.geometry.intersects(fz).values if len(r) else []
    r["near"] = (~r["hit"]) & r.geometry.intersects(near).values if len(r) else []
    r["is_bridge"] = r["bridge"].notna() & (r["bridge"].astype(str) != "no")
    r["len_hit_m"] = [g.intersection(fz).length if h else 0.0 for g, h in zip(r.geometry, r["hit"])] if len(r) else []
    br = r[r.is_bridge]
    stats = dict(buildings_total=int(len(b)), buildings_hit=int(b["hit"].sum()), buildings_at_risk=int(b["at_risk"].sum()),
                 road_km_total=round(float(r.length.sum()) / 1000, 1), road_km_hit=round(float(r["len_hit_m"].sum()) / 1000, 2),
                 bridges_total=int(len(br)), bridges_hit=int(br["hit"].sum()), bridges_at_risk=int(br["near"].sum()))
    return b, r, fz, stats

## 6. Q3 – Who is cut off? (road graph before vs after)

In [6]:
def build_graph(roads):
    G = nx.Graph()
    for geom in roads.geometry:
        for line in (geom.geoms if geom.geom_type == "MultiLineString" else [geom]):
            c = [(round(x, 1), round(y, 1)) for x, y in line.coords]
            for a, b in zip(c[:-1], c[1:]):
                if a != b:
                    G.add_edge(a, b, length=math.dist(a, b), seg=LineString([a, b]))
    return G

def remove_flooded_edges(G, fz):
    G2 = G.copy()
    ed = list(G2.edges(data=True))
    if not ed or fz.is_empty: return G2
    segs = np.array([d["seg"] for _, _, d in ed], dtype=object)
    hit = shapely.intersects(segs, fz)
    G2.remove_edges_from([(u, v) for (u, v, _), h in zip(ed, hit) if h])
    return G2

def cutoff_analysis(G_pre, G_post, osm, fz, snap_tol=500):
    nodes = np.array(list(G_pre.nodes)); tree = cKDTree(nodes)
    def snap(gdf):
        c = gdf.geometry.centroid
        d, i = tree.query(np.c_[c.x, c.y])
        return [tuple(nodes[j]) if dd <= snap_tol else None for dd, j in zip(d, i)]
    pl = osm["places"].copy()
    pl["node"] = snap(pl)
    hosp = [n for n in snap(osm["hospitals"]) if n is not None] if len(osm["hospitals"]) else []
    towns = osm["places"][osm["places"]["place"].isin(["town", "city"])]
    town_nodes = [n for n in snap(towns) if n is not None] if len(towns) else []
    def dists(G, src):
        return nx.multi_source_dijkstra_path_length(G, src, weight="length") if src else {}
    D = {k: (dists(G_pre, s), dists(G_post, s)) for k, s in [("hospital", hosp), ("town", town_nodes)]}
    in_flood = pl.geometry.intersects(fz.buffer(100)).values if len(pl) else []
    pl["in_flood"] = in_flood
    for k, (pre, post) in D.items():
        stat, dpre, dpost = [], [], []
        for n in pl["node"]:
            a = pre.get(n) if n else None; b = post.get(n) if n else None
            dpre.append(None if a is None else round(a / 1000, 1)); dpost.append(None if b is None else round(b / 1000, 1))
            if n is None: stat.append("no_road_data")
            elif a is None: stat.append("not_connected_before")
            elif b is None: stat.append("CUT_OFF")
            elif b > 1.5 * a + 500: stat.append("detour")
            else: stat.append("ok")
        pl[f"{k}_status"], pl[f"{k}_km_pre"], pl[f"{k}_km_post"] = stat, dpre, dpost
    return pl

## 7. Bonus – trace the flood path downstream from an upstream point (Copernicus DEM)

In [7]:
def trace_flood_path(dem, grid, lonlat, places, corridor_m=500, max_steps=40000):
    x, y = grid.from_ll.transform(*lonlat)
    c = int((x - grid.bounds[0]) / grid.res); r = int((grid.bounds[3] - y) / grid.res)
    H, W = dem.shape
    if not (0 <= r < H and 0 <= c < W): raise ValueError("Upstream point is outside the AOI")
    z = ndi.gaussian_filter(dem, 1.5)
    r0, r1, c0, c1 = max(r - 5, 0), min(r + 6, H), max(c - 5, 0), min(c + 6, W)
    sub = z[r0:r1, c0:c1]; k = np.unravel_index(np.argmin(sub), sub.shape); r, c = r0 + k[0], c0 + k[1]
    path, seen = [(r, c)], {(r, c)}
    for _ in range(max_steps):
        cand = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                n = (r + dr, c + dc)
                if (dr or dc) and 0 <= n[0] < H and 0 <= n[1] < W and n not in seen:
                    cand.append((z[n] - z[r, c], n))
        if not cand: break
        _, (r, c) = min(cand, key=lambda t: t[0] / (1.414 if t[1][0] != r and t[1][1] != c else 1.0))
        path.append((r, c)); seen.add((r, c))
        if r in (0, H - 1) or c in (0, W - 1): break
    xy = np.array([(grid.bounds[0] + (c_ + .5) * grid.res, grid.bounds[3] - (r_ + .5) * grid.res) for r_, c_ in path])
    cum = np.r_[0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]
    cen = places.geometry.centroid; d, i = cKDTree(xy).query(np.c_[cen.x, cen.y])
    t = places.assign(km_downstream=np.round(cum[i] / 1000, 1), dist_to_path_m=np.round(d))
    t = t[t.dist_to_path_m <= corridor_m].sort_values("km_downstream")
    return LineString(xy), t[["name", "place", "km_downstream", "dist_to_path_m"]]

## 8. Facts -> Situation-report copilot (English + Nepali)
The LLM receives ONLY a JSON of numbers computed by the system, and every number in its reply is
validated against that JSON. If validation fails twice, a deterministic template is used instead.

In [8]:
NUM = re.compile(r"\d+(?:\.\d+)?")
MODEL = "claude-sonnet-5-5"

def build_facts(C, info, stats, flood_area, places, path_table):
    names = lambda col: sorted(places.loc[places[col] == "CUT_OFF", "name"].dropna().tolist())
    return {
        "event_date": C["flood_date"], "osm_snapshot": C["osm_date"],
        "imagery": info,
        "flood_area_km2": flood_area,
        "damage": stats,
        "settlements": {"total": int(len(places)),
                        "cut_off_from_hospital": names("hospital_status"),
                        "cut_off_from_town": names("town_status"),
                        "detour_to_hospital": sorted(places.loc[places["hospital_status"] == "detour", "name"].dropna().tolist()),
                        "inside_flood_zone": sorted(places.loc[places["in_flood"], "name"].dropna().tolist())},
        "flood_path_settlements": [] if path_table is None else path_table.fillna("").to_dict("records"),
        "limitations": ["Sentinel satellites revisit every few days; no early warning.",
                        "Damage is inferred from overlap with the flood mask, not field-verified.",
                        "OSM is incomplete in remote areas; roads missing from OSM cannot be analysed."],
    }

def numbers_ok(text, facts):
    allowed = {float(x) for x in NUM.findall(json.dumps(facts, ensure_ascii=False))}
    txt = re.sub(r"(?<=\d),(?=\d{3})", "", text)
    bad = [x for x in NUM.findall(txt) if float(x) not in allowed]
    return bad

def template_report(f, lang="en"):
    d, s = f["damage"], f["settlements"]
    if lang == "ne":
        return (f"बाढी मिति {f['event_date']}। उपग्रह आधारित अनुमान: प्रभावित क्षेत्र {f['flood_area_km2']} वर्ग किमी। "
                f"भवन प्रभावित {d['buildings_hit']} (जोखिममा {d['buildings_at_risk']}), सडक {d['road_km_hit']} किमी, "
                f"पुल प्रभावित {d['bridges_hit']}। अस्पतालसँग सडक सम्पर्क टुटेका बस्ती: "
                f"{', '.join(s['cut_off_from_hospital']) or 'कुनै छैन'}। यो शैक्षिक प्रोटोटाइप हो; क्षेत्र जाँच आवश्यक छ।")
    return (f"Flood date {f['event_date']}. Satellite-derived estimate: flooded/debris area {f['flood_area_km2']} km2. "
            f"Buildings hit {d['buildings_hit']} (at risk {d['buildings_at_risk']}); road length hit {d['road_km_hit']} km; "
            f"bridges hit {d['bridges_hit']}. Settlements cut off from a hospital: "
            f"{', '.join(s['cut_off_from_hospital']) or 'none detected'}. Educational prototype; field verification required.")

SYSTEM = ("You are a situation-report assistant for disaster-response teams. Use ONLY the JSON FACTS provided. "
          "Never invent, estimate, round, sum or derive any number: copy numbers exactly as they appear in FACTS. "
          "If FACTS do not contain the answer, say so. State that results are satellite-derived estimates from an "
          "educational prototype. Do not number list items. Use Western digits 0-9 even in Nepali. "
          "Be concise, factual and respectful; never speculate about victims.")

def llm(question, facts, lang):
    import anthropic
    cl = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    language = {"en": "English", "ne": "Nepali (Devanagari script)"}[lang]
    msg = f"FACTS:\n{json.dumps(facts, ensure_ascii=False)}\n\nRespond in {language}.\nTASK: {question}"
    for attempt in range(3):
        r = cl.messages.create(model=MODEL, max_tokens=900, system=SYSTEM, messages=[{"role": "user", "content": msg}])
        text = "".join(b.text for b in r.content if b.type == "text")
        bad = numbers_ok(text, facts)
        if not bad: return text
        msg += f"\n\nYour previous answer contained numbers not in FACTS: {bad}. Rewrite using only numbers from FACTS."
    return None

def ask(question, facts, lang="en"):
    """Chat with the copilot. Falls back to a template for report requests if no key / validation fails."""
    if not ANTHROPIC_KEY:
        return "(No ANTHROPIC_API_KEY set)\n" + template_report(facts, lang)
    out = llm(question, facts, lang)
    return out or ("(LLM output failed number validation; using template)\n" + template_report(facts, lang))

def situation_report(facts):
    q = ("Write a one-page situation report with sections: Summary, Flood extent, Damage, Cut-off settlements, "
         "Flood-path settlements (if any), Limitations. Max 250 words.")
    return {lang: ask(q, facts, lang) for lang in ("en", "ne")}

### Copilot LLM = Groq (free). Overrides the earlier llm() function.

In [9]:
GROQ_MODEL = "llama-3.3-70b-versatile"   # change if Groq retires it

def llm(question, facts, lang):
    language = {"en": "English", "ne": "Nepali (Devanagari script)"}[lang]
    msg = f"FACTS:\n{json.dumps(facts, ensure_ascii=False)}\n\nRespond in {language}.\nTASK: {question}"
    for attempt in range(3):
        r = requests.post("https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_KEY}"},
            json={"model": GROQ_MODEL, "temperature": 0.2, "max_tokens": 1200,
                  "messages": [{"role": "system", "content": SYSTEM},
                               {"role": "user", "content": msg}]}, timeout=120)
        if r.status_code != 200:
            print("Groq error:", r.status_code, r.text[:300]); return None
        text = r.json()["choices"][0]["message"]["content"]
        bad = numbers_ok(text, facts)
        if not bad: return text
        msg += f"\n\nYour previous answer contained numbers not in FACTS: {bad}. Rewrite using only numbers from FACTS."
    return None

## 9. Dashboard + report outputs

In [10]:
ATTR = ("Contains modified Copernicus Sentinel data 2026. Produced using Copernicus WorldDEM-30 © DLR e.V. 2010–2014 and "
        "© Airbus Defence and Space GmbH 2014–2018 provided under COPERNICUS by the European Union and ESA; all rights reserved. "
        "© OpenStreetMap contributors.")

def ll(gdf): return gdf.to_crs(4326)

def make_dashboard(grid, C, flood, b, r, places, hosp, flood_line, path):
    cx, cy = grid.to_ll.transform((grid.bounds[0] + grid.bounds[2]) / 2, (grid.bounds[1] + grid.bounds[3]) / 2)
    m = folium.Map(location=[cy, cx], zoom_start=11, tiles="CartoDB positron")
    color = {"water": "#1f78b4", "debris": "#8c5a2b"}
    for kind in ("water", "debris"):
        g = flood[flood.kind == kind]
        if len(g):
            folium.GeoJson(ll(g).to_json(), name=f"Flood: {kind}",
                           style_function=lambda _, c=color[kind]: dict(color=c, fillColor=c, fillOpacity=.55, weight=1)).add_to(m)
    bh = b[b.hit | b.at_risk]
    if len(bh):
        folium.GeoJson(ll(bh[["hit", "at_risk", "geometry"]]).to_json(), name="Buildings hit / at risk",
                       style_function=lambda f: dict(color="red" if f["properties"]["hit"] else "orange", weight=1, fillOpacity=.7)).add_to(m)
    rh = r[r.hit]
    if len(rh):
        folium.GeoJson(ll(rh[["highway", "is_bridge", "geometry"]]).to_json(), name="Roads/bridges hit",
                       style_function=lambda f: dict(color="purple" if f["properties"]["is_bridge"] else "red", weight=4)).add_to(m)
    fg = folium.FeatureGroup(name="Settlements").add_to(m)
    for _, p in ll(places).iterrows():
        st = p["hospital_status"]
        col = {"CUT_OFF": "red", "detour": "orange", "ok": "green"}.get(st, "gray")
        folium.CircleMarker([p.geometry.y, p.geometry.x], radius=7, color=col, fill=True, fill_opacity=.9,
                            tooltip=f"{p['name']} | hospital: {st} | town: {p['town_status']}").add_to(fg)
    fh = folium.FeatureGroup(name="Hospitals").add_to(m)
    for _, h in ll(hosp).iterrows():
        pt = h.geometry.centroid
        folium.Marker([pt.y, pt.x], icon=folium.Icon(color="black", icon="plus"), tooltip=h.get("name") or "hospital").add_to(fh)
    if flood_line is not None:
        folium.GeoJson(gpd.GeoSeries([flood_line], crs=grid.epsg).to_crs(4326).to_json(), name="Traced flood path",
                       style_function=lambda _: dict(color="cyan", weight=3, dashArray="6")).add_to(m)
    folium.LayerControl().add_to(m)
    m.get_root().html.add_child(folium.Element(
        f"<div style='position:fixed;bottom:0;left:0;right:0;z-index:9999;background:#fffc;font:11px sans-serif;padding:3px'>"
        f"{ATTR} Educational prototype – not an operational tool.</div>"))
    m.save(path)
    return m

def make_report_html(facts, texts, png_b64, path):
    esc = lambda s: s.replace("&", "&amp;").replace("<", "&lt;").replace("\n", "<br>")
    d = facts["damage"]
    html = f"""<!doctype html><html><head><meta charset='utf-8'><title>Situation report</title>
<style>body{{font:12px/1.4 'Noto Sans','Noto Sans Devanagari',sans-serif;max-width:800px;margin:auto}}
td,th{{border:1px solid #999;padding:2px 6px}}table{{border-collapse:collapse}}img{{max-width:100%}}
.f{{font-size:9px;color:#555}}</style></head><body>
<h2>Flood situation report – {facts['event_date']}</h2>
<img src='data:image/png;base64,{png_b64}'>
<table><tr><th>Flooded/debris area</th><th>Buildings hit</th><th>At risk</th><th>Road km hit</th><th>Bridges hit</th></tr>
<tr><td>{facts['flood_area_km2']} km²</td><td>{d['buildings_hit']}</td><td>{d['buildings_at_risk']}</td><td>{d['road_km_hit']}</td><td>{d['bridges_hit']}</td></tr></table>
<h3>English</h3><p>{esc(texts['en'])}</p><h3>नेपाली</h3><p>{esc(texts['ne'])}</p>
<p class='f'>{ATTR}<br>Educational prototype; not an operational tool. Estimates are not field-verified.</p></body></html>"""
    open(path, "w", encoding="utf-8").write(html)

def static_png(grid, cls, perm):
    rgb = np.ones(cls.shape + (3,)); rgb[perm] = (.75, .85, .95)
    rgb[cls == 1] = (.12, .47, .71); rgb[cls == 2] = (.55, .35, .17)
    fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(rgb); ax.axis("off")
    ax.set_title("Water (blue) / debris-change (brown)")
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

## 10. Evaluation against EMSR927 (CHECKING ONLY – never use as input)

In [11]:
def evaluate_against_reference(flood, ref_path, grid):
    """ref_path: GeoJSON/Shapefile of the official EMS flood/debris extent you downloaded manually."""
    ref = gpd.read_file(ref_path).to_crs(grid.epsg)
    rast = lambda g: features.rasterize([(x, 1) for x in g.geometry if x is not None], out_shape=(grid.H, grid.W),
                                        transform=grid.transform, dtype="uint8").astype(bool)
    p, t = rast(flood), rast(ref)
    tp = (p & t).sum(); fp = (p & ~t).sum(); fn = (~p & t).sum()
    res = dict(IoU=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))
    res["F1"] = 2 * res["precision"] * res["recall"] / max(res["precision"] + res["recall"], 1e-9)
    return {k: round(float(v), 3) for k, v in res.items()}

In [12]:
from rasterio.warp import reproject, Resampling

def get_dem(grid):
    """Copernicus DEM GLO-30 from the public AWS bucket (no key), resampled onto our grid."""
    w, s, e, n = CONFIG["bbox"]
    dem = np.full((grid.H, grid.W), np.nan, "float32")
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        for lat in range(math.floor(s), math.floor(n) + 1):
            for lon in range(math.floor(w), math.floor(e) + 1):
                ns = f"N{lat:02d}" if lat >= 0 else f"S{-lat:02d}"
                ew = f"E{lon:03d}" if lon >= 0 else f"W{-lon:03d}"
                name = f"Copernicus_DSM_COG_10_{ns}_00_{ew}_00_DEM"
                url = f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif"
                try:
                    with rasterio.open(url) as src:
                        tmp = np.full(dem.shape, np.nan, "float32")
                        reproject(rasterio.band(src, 1), tmp, dst_transform=grid.transform,
                                  dst_crs=f"EPSG:{grid.epsg}", dst_nodata=np.nan,
                                  resampling=Resampling.bilinear)
                    dem = np.where(np.isnan(dem), tmp, dem)
                except Exception as ex:
                    print("DEM tile skipped:", name, ex)
    if np.isnan(dem).all():
        raise RuntimeError("No DEM tiles could be read - check internet access / AOI")
    dem[np.isnan(dem)] = np.nanmedian(dem)
    return dem

In [13]:
from shapely.geometry import Point, Polygon

UA = {"User-Agent": "floodwatch-hackathon/1.0 (educational prototype)"}
OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter"]

def ohsome(bbox, flt, time_, cols):
    r = requests.post(OHSOME_URL, headers=UA, data={"bboxes": ",".join(map(str, bbox)), "time": time_,
                      "filter": flt, "properties": "tags"}, timeout=900)
    r.raise_for_status()
    fc = r.json().get("features", [])
    gdf = gpd.GeoDataFrame.from_features(fc, crs=4326) if fc else gpd.GeoDataFrame(geometry=[], crs=4326)
    for c in cols:
        if c not in gdf: gdf[c] = None
    return gdf

def overpass(bbox, body, date, cols, geom):
    w, s, e, n = bbox
    q = (f'[out:json][timeout:600][date:"{date}T00:00:00Z"];({body.replace("@@", f"({s},{w},{n},{e})")});'
         + ("out center tags;" if geom == "center" else "out geom tags;"))
    last = None
    for url in OVERPASS:
        try:
            r = requests.post(url, data={"data": q}, headers=UA, timeout=900)
            if r.status_code == 200:
                js = r.json(); break
            last = f"{url} -> {r.status_code}"
        except Exception as ex:
            last = f"{url} -> {ex}"
    else:
        raise RuntimeError("All Overpass servers failed: " + str(last))
    rows = []
    for el in js.get("elements", []):
        tags = el.get("tags", {}); g = None
        if el["type"] == "node" and "lat" in el: g = Point(el["lon"], el["lat"])
        elif geom == "center" and "center" in el: g = Point(el["center"]["lon"], el["center"]["lat"])
        elif el["type"] == "way" and "geometry" in el:
            pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
            if geom == "poly":
                if len(pts) >= 4 and pts[0] == pts[-1]: g = Polygon(pts)
            elif len(pts) >= 2: g = LineString(pts)
        if g is not None:
            rows.append({**{c: tags.get(c) for c in cols}, "geometry": g})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
           gpd.GeoDataFrame({c: [] for c in cols}, geometry=[], crs=4326)

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    rc = ROAD_CLASSES.replace(",", "|")
    L = {
     "buildings": ("building=* and geometry:polygon", ["building", "name"], 'way["building"]@@;', "poly"),
     "roads": (f"highway in ({ROAD_CLASSES}) and geometry:line", ["highway", "bridge", "name"],
               f'way["highway"~"^({rc})$"]@@;', "line"),
     "places": ("place in (city,town,village,hamlet) and geometry:point", ["place", "name"],
                'node["place"~"^(city|town|village|hamlet)$"]@@;', "point"),
     "hospitals": ("(amenity=hospital or healthcare=hospital) and (geometry:point or geometry:polygon)", ["name"],
                   'nwr["amenity"="hospital"]@@;nwr["healthcare"="hospital"]@@;', "center"),
    }
    osm = {}
    for k, (flt, cols, op, geom) in L.items():
        try:
            osm[k] = ohsome(bb, flt, t, cols); print(k, "via ohsome:", len(osm[k]))
        except Exception as ex:
            print(k, "ohsome failed (", str(ex)[:60], ") -> Overpass")
            osm[k] = overpass(bb, op, t, cols, geom); print(k, "via Overpass:", len(osm[k]))
    if osm["hospitals"].empty:
        print("No hospitals in AOI -> falling back to clinics")
        osm["hospitals"] = overpass(bb, 'nwr["amenity"="clinic"]@@;', t, ["name"], "center")
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}

In [14]:
UA = {"User-Agent": "floodwatch-hackathon/1.0 (educational prototype)"}
OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter",
            "https://overpass.private.coffee/api/interpreter"]

# remember the flood polygons so buildings are only fetched near them
_orig_vectorize = vectorize
def vectorize(cls, grid):
    g = _orig_vectorize(cls, grid)
    globals()["_LAST_FLOOD"] = g
    return g

def op_fetch(bb, body, date, cols, geom):
    w, s, e, n = bb
    q = (f'[out:json][timeout:180][date:"{date}T00:00:00Z"];({body.replace("@@", f"({s},{w},{n},{e})")});'
         + ("out center tags;" if geom == "center" else "out geom tags;"))
    for attempt in range(3):
        for url in OVERPASS:
            try:
                r = requests.post(url, data={"data": q}, headers=UA, timeout=240)
                if r.status_code == 200:
                    rows = []
                    for el in r.json().get("elements", []):
                        tags = el.get("tags", {}); g = None
                        if el["type"] == "node" and "lat" in el: g = Point(el["lon"], el["lat"])
                        elif geom == "center" and "center" in el: g = Point(el["center"]["lon"], el["center"]["lat"])
                        elif el["type"] == "way" and "geometry" in el:
                            pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
                            if geom == "poly":
                                if len(pts) >= 4 and pts[0] == pts[-1]: g = Polygon(pts)
                            elif len(pts) >= 2: g = LineString(pts)
                        if g is not None:
                            rows.append({**{c: tags.get(c) for c in cols}, "osmid": f'{el["type"]}/{el["id"]}', "geometry": g})
                    return rows
                print("   ", url.split("/")[2], r.status_code)
            except Exception as ex:
                print("   ", url.split("/")[2], str(ex)[:60])
        time.sleep(5 * (attempt + 1))
    raise RuntimeError("Overpass failed for a tile after retries")

def make_tiles(bb, step):
    w, s, e, n = bb
    xs = np.arange(w, e, step); ys = np.arange(s, n, step)
    return [(x, y, min(x + step, e), min(y + step, n)) for x in xs for y in ys]

def op_layer(tiles, body, date, cols, geom):
    rows = []
    for i, tb in enumerate(tiles):
        rows += op_fetch(tb, body, date, cols, geom)
        print(f"   tile {i+1}/{len(tiles)} ok, total {len(rows)}")
        time.sleep(1)
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
          gpd.GeoDataFrame({c: [] for c in cols + ["osmid"]}, geometry=[], crs=4326)
    return gdf.drop_duplicates("osmid") if len(gdf) else gdf

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    rc = ROAD_CLASSES.replace(",", "|")
    try:   # quick test whether ohsome accepts us
        ohsome((85.3, 28.1, 85.31, 28.11), "place=*", t, ["name"]); use_ohsome = True
    except Exception as ex:
        print("ohsome unavailable (", str(ex)[:50], ") -> using Overpass"); use_ohsome = False
    big = make_tiles(bb, 0.12)
    osm = {}
    # roads, places, hospitals: whole area
    if use_ohsome:
        osm["roads"] = ohsome(bb, f"highway in ({ROAD_CLASSES}) and geometry:line", t, ["highway", "bridge", "name"])
        osm["places"] = ohsome(bb, "place in (city,town,village,hamlet) and geometry:point", t, ["place", "name"])
        osm["hospitals"] = ohsome(bb, "(amenity=hospital or healthcare=hospital) and (geometry:point or geometry:polygon)", t, ["name"])
    else:
        print("roads..."); osm["roads"] = op_layer(big, f'way["highway"~"^({rc})$"]@@;', t, ["highway", "bridge", "name"], "line")
        print("places..."); osm["places"] = op_layer(big, 'node["place"~"^(city|town|village|hamlet)$"]@@;', t, ["place", "name"], "point")
        print("hospitals..."); osm["hospitals"] = op_layer(big, 'nwr["amenity"="hospital"]@@;nwr["healthcare"="hospital"]@@;', t, ["name"], "center")
    if osm["hospitals"].empty:
        print("No hospitals in AOI -> trying clinics")
        osm["hospitals"] = op_layer(big, 'nwr["amenity"="clinic"]@@;', t, ["name"], "center")
    # buildings: only tiles close to the flood zone
    fl = globals().get("_LAST_FLOOD")
    if fl is None or len(fl) == 0:
        print("No flood polygons -> skipping buildings"); osm["buildings"] = gpd.GeoDataFrame({"building": [], "name": []}, geometry=[], crs=4326)
    else:
        zone = shapely.union_all(fl.to_crs(4326).geometry.values).buffer(0.01)
        small = [tb for tb in make_tiles(bb, 0.03) if box(*tb).intersects(zone)][:60]
        print(f"buildings: {len(small)} tiles near flood zone...")
        if use_ohsome:
            osm["buildings"] = ohsome(bb, "building=* and geometry:polygon", t, ["building", "name"])
        else:
            osm["buildings"] = op_layer(small, 'way["building"]@@;', t, ["building", "name"], "poly")
    for k, v in osm.items(): print(f"  {k}: {len(v)}")
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}

In [15]:
# smaller area = fewer tiles (adjust if you want)
CONFIG["bbox"] = (85.25, 28.05, 85.45, 28.30)

_OPCACHE = globals().get("_OPCACHE", {})
FAILED_TILES = []

def _op_once(tb, body, date, cols, geom, tries=2):
    key = (body, date, tuple(round(x, 4) for x in tb), geom)
    if key in _OPCACHE: return _OPCACHE[key]
    w, s, e, n = tb
    q = (f'[out:json][timeout:120][date:"{date}T00:00:00Z"];({body.replace("@@", f"({s},{w},{n},{e})")});'
         + ("out center tags;" if geom == "center" else "out geom tags;"))
    for attempt in range(tries):
        for url in OVERPASS:
            try:
                r = requests.post(url, data={"data": q}, headers=UA, timeout=150)
                if r.status_code == 200:
                    rows = []
                    for el in r.json().get("elements", []):
                        tags = el.get("tags", {}); g = None
                        if el["type"] == "node" and "lat" in el: g = Point(el["lon"], el["lat"])
                        elif geom == "center" and "center" in el: g = Point(el["center"]["lon"], el["center"]["lat"])
                        elif el["type"] == "way" and "geometry" in el:
                            pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
                            if geom == "poly":
                                if len(pts) >= 4 and pts[0] == pts[-1]: g = Polygon(pts)
                            elif len(pts) >= 2: g = LineString(pts)
                        if g is not None:
                            rows.append({**{c: tags.get(c) for c in cols}, "osmid": f'{el["type"]}/{el["id"]}', "geometry": g})
                    _OPCACHE[key] = rows
                    return rows
            except Exception:
                pass
        time.sleep(8 * (attempt + 1))
    return None

def op_layer(tiles, body, date, cols, geom):
    rows, pending = [], []
    for i, tb in enumerate(tiles):
        r = _op_once(tb, body, date, cols, geom)
        if r is None: pending.append(tb); print(f"   tile {i+1}/{len(tiles)} busy, will retry later")
        else: rows += r; print(f"   tile {i+1}/{len(tiles)} ok, total {len(rows)}")
        time.sleep(1)
    for rnd in range(2):                      # retry failed tiles after the servers calm down
        if not pending: break
        print(f"   retrying {len(pending)} tile(s) in 30 s ..."); time.sleep(30)
        still = []
        for tb in pending:
            r = _op_once(tb, body, date, cols, geom, tries=3)
            if r is None: still.append(tb)
            else: rows += r; print("   retried tile ok, total", len(rows))
        pending = still
    if pending:
        FAILED_TILES.extend(pending)
        print(f"   WARNING: {len(pending)} tile(s) missing -> coverage incomplete (mention in limitations)")
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
          gpd.GeoDataFrame({c: [] for c in cols + ["osmid"]}, geometry=[], crs=4326)
    return gdf.drop_duplicates("osmid") if len(gdf) else gdf

_mt = make_tiles
def make_tiles(bb, step): return _mt(bb, min(step, 0.06))
from shapely.geometry import box

In [16]:
# ---- PATCHES (override earlier definitions) ----
def s1_db(arr):
    """(VV, VH, mask) -> speckle-reduced dB. Non-finite/absurd values excluded BEFORE filtering."""
    vv, vh, m = arr
    valid = ((m > 0) & np.isfinite(vv) & np.isfinite(vh)
             & (vv > 0) & (vh > 0) & (vv < 10) & (vh < 10))
    den = ndi.uniform_filter(valid.astype("float64"), 5)
    out = []
    for b in (vv, vh):
        x = np.where(valid, b, 0.0).astype("float64")
        num = ndi.uniform_filter(x, 5)
        lin = np.where(den > 0.3, num / np.maximum(den, 1e-6), np.nan)
        out.append(10 * np.log10(np.clip(lin, 1e-5, None)))
    return out[0], out[1]

GROQ_MODEL = "openai/gpt-oss-120b"

def llm(question, facts, lang):
    language = {"en": "English", "ne": "Nepali (Devanagari script)"}[lang]
    msg = f"FACTS:\n{json.dumps(facts, ensure_ascii=False)}\n\nRespond in {language}.\nTASK: {question}"
    for attempt in range(3):
        r = requests.post("https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_KEY}"},
            json={"model": GROQ_MODEL, "temperature": 0.2, "max_tokens": 3000,
                  "messages": [{"role": "system", "content": SYSTEM},
                               {"role": "user", "content": msg}]}, timeout=120)
        if r.status_code != 200:
            print("Groq error:", r.status_code, r.text[:300]); return None
        text = r.json()["choices"][0]["message"].get("content") or ""
        if not text.strip(): continue
        bad = numbers_ok(text, facts)
        if not bad: return text
        msg += f"\n\nYour previous answer contained numbers not in FACTS: {bad}. Rewrite using only numbers from FACTS."
    return None

In [17]:
import heapq

def fill_depressions(dem, eps=1e-3):
    H, W = dem.shape
    z = dem.astype("float64").copy()
    closed = np.zeros((H, W), bool); heap = []
    for r in range(H):
        for c in (0, W - 1):
            heapq.heappush(heap, (z[r, c], r, c)); closed[r, c] = True
    for c in range(1, W - 1):
        for r in (0, H - 1):
            heapq.heappush(heap, (z[r, c], r, c)); closed[r, c] = True
    nb = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    while heap:
        h, r, c = heapq.heappop(heap)
        for dr, dc in nb:
            rr, cc = r + dr, c + dc
            if 0 <= rr < H and 0 <= cc < W and not closed[rr, cc]:
                closed[rr, cc] = True
                if z[rr, cc] <= h: z[rr, cc] = h + eps
                heapq.heappush(heap, (z[rr, cc], rr, cc))
    return z

def trace_flood_path(dem, grid, lonlat, places, corridor_m=500, max_steps=200000):
    x, y = grid.from_ll.transform(*lonlat)
    c = int((x - grid.bounds[0]) / grid.res); r = int((grid.bounds[3] - y) / grid.res)
    H, W = dem.shape
    if not (0 <= r < H and 0 <= c < W): raise ValueError("Upstream point is outside the AOI")
    z = fill_depressions(ndi.gaussian_filter(dem, 1.0))
    r0, r1, c0, c1 = max(r-10, 0), min(r+11, H), max(c-10, 0), min(c+11, W)
    sub = z[r0:r1, c0:c1]; k = np.unravel_index(np.argmin(sub), sub.shape); r, c = r0 + k[0], c0 + k[1]
    path = [(r, c)]
    for _ in range(max_steps):
        best, bs = None, 0.0
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if not (dr or dc): continue
                rr, cc = r + dr, c + dc
                if 0 <= rr < H and 0 <= cc < W:
                    s = (z[r, c] - z[rr, cc]) / (1.414 if dr and dc else 1.0)
                    if s > bs: bs, best = s, (rr, cc)
        if best is None: break
        r, c = best; path.append(best)
        if r in (0, H-1) or c in (0, W-1): break
    xy = np.array([(grid.bounds[0] + (c_ + .5) * grid.res, grid.bounds[3] - (r_ + .5) * grid.res) for r_, c_ in path])
    cum = np.r_[0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]
    cen = places.geometry.centroid; d, i = cKDTree(xy).query(np.c_[cen.x, cen.y])
    t = places.assign(km_downstream=np.round(cum[i] / 1000, 1), dist_to_path_m=np.round(d))
    t = t[t.dist_to_path_m <= corridor_m].sort_values("km_downstream")
    return LineString(xy), t[["name", "place", "km_downstream", "dist_to_path_m"]]

In [18]:
def hand_proxy(dem, seeds):
    return np.zeros_like(dem)   # HAND is unreliable in this gorge; rely on slope + radar change instead

## 11. Run everything end-to-end

In [19]:
if "_bf_base" not in globals():
    _bf_base = build_facts          # capture the current version once

def build_facts(C, info, stats, flood_area, places, path_table):
    f = _bf_base(C, info, stats, flood_area, places, path_table)
    ok = ["ok", "detour", "CUT_OFF"]
    s = f["settlements"]
    s["analysed_for_hospital_access"] = int(places["hospital_status"].isin(ok).sum())
    s["no_road_data_within_500m"] = int((places["hospital_status"] == "no_road_data").sum())
    if not places["town_status"].isin(ok).any():
        s["cut_off_from_town"] = "not assessed: no town found in OSM data for this area"
    kinds = sorted(set(globals().get("_LAST_FLOOD", gpd.GeoDataFrame({"kind": []})).get("kind", [])))
    f["flood_mask_kinds"] = kinds
    if "water" not in kinds:
        f["flood_mask_note"] = "Mask contains radar-change (debris-affected) areas only; no open water was detected."
    if not any("single route" in x for x in f["limitations"]):
        f["limitations"] += [
            "Cut-off status depends on a few road segments in OpenStreetMap, which here is close to a single route; one flagged segment can isolate a whole branch.",
            "Results are sensitive to detection thresholds: more permissive settings give a much larger affected area and more cut-off settlements.",
            "No town was found in the OSM data for this area, so access to a town was not assessed.",
        ]
    return f

if "_sys_final" not in globals():
    _sys_final = True
    SYSTEM = SYSTEM + (" Report overlap with the detected mask as 'intersects the detected mask', not as impact or damage. "
                       "If a FACTS field says 'not assessed', say it was not assessed; never say 'none' for it. "
                       "Include EVERY item of FACTS limitations. Always finish the Limitations section.")

In [20]:
CONFIG.update(change_db=3.0, min_blob_px=8, max_slope_deg=25.0, max_hand_m=60.0)
CONFIG["bbox"] = (85.25, 28.05, 85.45, 28.30)
print({k: CONFIG[k] for k in ("bbox", "change_db", "min_blob_px", "max_slope_deg")})

{'bbox': (85.25, 28.05, 85.45, 28.3), 'change_db': 3.0, 'min_blob_px': 8, 'max_slope_deg': 25.0}


In [21]:
def remove_flooded_edges(G, fz, min_len_m=30):
    G2 = G.copy()
    ed = list(G2.edges(data=True))
    if not ed or fz.is_empty: return G2
    shapely.prepare(fz)
    segs = np.array([d["seg"] for _, _, d in ed], dtype=object)
    hit = shapely.intersects(segs, fz)
    L = np.zeros(len(ed))
    L[hit] = shapely.length(shapely.intersection(segs[hit], fz))
    G2.remove_edges_from([(u, v) for (u, v, _), l in zip(ed, L) if l >= min_len_m])
    return G2

In [22]:
# ---- final report-prompt rules (guarded so re-running does not stack them) ----
if "_sys_final2" not in globals():
    _sys_final2 = True
    SYSTEM = SYSTEM + (" Make NO claims beyond what FACTS state: no statements about causes, severity, disruption or impacts. "
                       "Describe the flood area as 'flooded or debris-affected area'; say 'water' only if FACTS flood_mask_kinds contains 'water'. "
                       "Do not describe the result as complete; it is a detected minimum. "
                       "In Nepali, translate place types (village = गाउँ, hamlet = बस्ती).")

# headline settings (chosen BEFORE looking at any reference data)
CONFIG.update(change_db=3.0, min_blob_px=8, max_slope_deg=25.0, max_hand_m=60.0)
CONFIG["bbox"] = (85.25, 28.05, 85.45, 28.30)
print({k: CONFIG[k] for k in ("bbox", "change_db", "min_blob_px", "max_slope_deg")})

{'bbox': (85.25, 28.05, 85.45, 28.3), 'change_db': 3.0, 'min_blob_px': 8, 'max_slope_deg': 25.0}


## 10b. Refinement patch (small, structural; overrides the functions above)
* **Accuracy:** two smoothing scales (3x3 and 5x5) instead of 5x5 only, no 3x3 opening (it deleted flow paths narrower than 90 m), median-offset correction, VH must agree with VV, slope limit 35 degrees, Sentinel-2 used wherever both dates are clear.
* **Speed:** disk cache for rasters and OSM, one Overpass query per layer instead of ~60 tiles, the redundant first `run()` removed.
* **Consistency:** settlement counts add up, 'town not assessed' is shown as such, cross-border names flagged, one deterministic EN/NE report plus an optional validated AI summary, quantified threshold sensitivity.
* **Honesty:** these changes are physical/structural, **not fitted to EMSR927**. Report both runs (first run IoU = 0, this run) in the report.

In [23]:
# ================= PATCH 1: detection fixes + caching =================
# Structural changes with a physical reason. They were NOT fitted to EMSR927 (the reference is for checking only).
import pickle, hashlib
CONFIG.update(
    bbox=(85.10, 27.90, 85.45, 28.32),  # whole Bhote Koshi-Trishuli corridor named in the problem statement
    smooth_scales=(3, 5),  # was 5 only (150 m window on the 30 m grid): narrow gorge flow paths were averaged away
    min_blob_px=6,       # was 8 plus a 3x3 opening that deleted anything narrower than 90 m: now only tiny specks go
    max_slope_deg=35.0,  # was 25: gorge floors look steep at 30 m resolution
    vh_confirm_db=0.5,   # fine (3x3) scale only: VH must change in the same direction as VV by >0.5 dB (cuts speckle false alarms)
    s2_min_clear=0.02,   # use Sentinel-2 wherever BOTH dates are clear (was: only if >15% of the whole scene was clear)
    osm_pad_deg=0.10,    # look for towns/hospitals/roads this far beyond the AOI (nearest hospital may lie outside it)
)

# ---- disk cache: re-runs and threshold checks no longer re-download rasters ----
CACHE_DIR = os.path.join(CONFIG["out_dir"], "cache"); os.makedirs(CACHE_DIR, exist_ok=True)
def cached(key, fn):
    p = os.path.join(CACHE_DIR, hashlib.md5(repr(key).encode()).hexdigest() + ".pkl")
    if os.path.exists(p):
        try: return pickle.load(open(p, "rb"))
        except Exception: pass
    v = fn()
    try: pickle.dump(v, open(p, "wb"))
    except Exception as ex: print("cache write skipped:", ex)
    return v

if "_s1_raw" not in globals():
    _s1_raw, _s2_raw, _dem_raw = s1_scene, s2_composite, get_dem
def s1_scene(grid, day, orbit):
    return cached(("s1", day, orbit, grid.bounds, grid.res), lambda: _s1_raw(grid, day, orbit))
def s2_composite(grid, t0, t1):
    return cached(("s2", t0, t1, grid.bounds, grid.res), lambda: _s2_raw(grid, t0, t1))
def get_dem(grid):
    return cached(("dem", grid.bounds, grid.res), lambda: _dem_raw(grid))

# ---- radar change detection ----
def s1_db(arr, k=None):
    """(VV, VH, mask) -> speckle-reduced dB on a k x k window (filter in linear power, invalid pixels excluded first)."""
    k = k or CONFIG.get("smooth_px", 3)
    vv, vh, m = arr
    valid = ((m > 0) & np.isfinite(vv) & np.isfinite(vh) & (vv > 0) & (vh > 0) & (vv < 10) & (vh < 10))
    den = ndi.uniform_filter(valid.astype("float64"), k)
    out = []
    for b in (vv, vh):
        x = np.where(valid, b, 0.0).astype("float64")
        lin = np.where(den > 0.3, ndi.uniform_filter(x, k) / np.maximum(den, 1e-6), np.nan)
        out.append(10 * np.log10(np.clip(lin, 1e-5, None)))
    return out[0], out[1]

def clean(mask, min_px):
    """Drop specks smaller than min_px (8-connected). No morphological opening: it erased narrow flow paths."""
    lab, n = ndi.label(mask, structure=np.ones((3, 3)))
    if n == 0: return mask
    size = np.bincount(lab.ravel()); keep = size >= min_px; keep[0] = False
    return ndi.binary_fill_holes(keep[lab])

def _detect_k(pre, post, slope, hand, C, k):
    vv0, vh0 = s1_db(pre, k); vv1, vh1 = s1_db(post, k)
    ok = np.isfinite(vv0) & np.isfinite(vv1) & np.isfinite(vh0) & np.isfinite(vh1)
    water_pre = vv0 < C["water_vv_db"]
    perm_water = water_pre & (slope < 5)
    shadow = vv0 < -22                                       # radar shadow / very dark before -> ignore
    zone = (slope < C["max_slope_deg"]) & (hand < C["max_hand_m"]) & ok
    new_water = (vv1 < C["water_vv_db"]) & (vh1 < C["water_vh_db"]) & ~water_pre & zone
    dv, dh = vv1 - vv0, vh1 - vh0
    if ok.any():                                             # remove scene-wide offset (soil moisture / gain differences)
        dv = dv - np.nanmedian(dv[ok]); dh = dh - np.nanmedian(dh[ok])
    agree = (np.sign(dv) == np.sign(dh)) & (np.abs(dh) > C.get("vh_confirm_db", 0.0)) if k <= 3 else True   # VH check only at the noisier fine scale
    debris = (np.abs(dv) > C["change_db"]) & agree & ~new_water & ~shadow & zone
    return new_water, debris, perm_water, ok

def detect_s1(pre, post, slope, hand, C):
    """Two smoothing scales: 3x3 catches narrow gorge flow paths, 5x5 keeps wide, weak changes. Union of both."""
    W_, D_, P_, K_ = None, None, None, None
    for k in C.get("smooth_scales", (3, 5)):
        w, d, p, ok = _detect_k(pre, post, slope, hand, C, k)
        W_ = w if W_ is None else W_ | w; D_ = d if D_ is None else D_ | d
        P_, K_ = (p, ok) if P_ is None else (P_, K_)
    return clean(W_, C["min_blob_px"]), clean(D_ & ~W_, C["min_blob_px"]), P_, K_

def map_flood(grid, C):
    dem = get_dem(grid); slope = terrain(dem, grid.res)
    pair = find_s1_pair(C["bbox"], C["flood_date"]); print("S1 pair:", pair)
    pre = s1_scene(grid, pair["pre_day"], pair["orbit"]); post = s1_scene(grid, pair["post_day"], pair["orbit"])
    hand = np.zeros_like(dem)        # HAND proxy stays off: unreliable in gorges; slope + radar change are used instead
    w1, d1, perm, ok = detect_s1(pre, post, slope, hand, C)
    gap = (dt.date.fromisoformat(pair["post_day"]) - dt.date.fromisoformat(pair["pre_day"])).days
    info = dict(s1_pre=pair["pre_day"], s1_post=pair["post_day"], s1_gap_days=gap, orbit=pair["orbit"],
                s1_valid_pct=round(100 * float(ok.mean()), 1), s2_used=False, s2_clear_pct=0.0)
    w2 = d2 = np.zeros_like(w1)
    if C["use_optical"]:
        try:
            fd = dt.date.fromisoformat(C["flood_date"])
            s2pre = s2_composite(grid, fd - dt.timedelta(days=45), fd - dt.timedelta(days=1))
            s2post = s2_composite(grid, fd + dt.timedelta(days=1), fd + dt.timedelta(days=20))
            w2, d2, frac = detect_s2(s2pre, s2post, slope, hand, C)       # already restricted to pixels clear on both dates
            use = frac > C.get("s2_min_clear", 0.02)
            info.update(s2_clear_pct=round(100 * frac, 1), s2_used=bool(use))
            if not use: w2 = d2 = np.zeros_like(w1)
        except Exception as e:
            print("Optical step skipped:", e)
    water = w1 | w2
    debris = (d1 | d2) & ~water
    cls = np.zeros(slope.shape, "uint8"); cls[debris] = 2; cls[water] = 1
    return dict(cls=cls, dem=dem, slope=slope, hand=hand, perm_water=perm, info=info, pre=pre, post=post)

def sensitivity(F, grid, C, levels=(2.0, 3.0, 4.0)):
    """Radar-only detected area for a few change thresholds (no reference data involved). Cheap: uses cached rasters."""
    out = {}
    for L in levels:
        w, d, _, _ = detect_s1(F["pre"], F["post"], F["slope"], F["hand"], dict(C, change_db=L))
        out[f"{L:.1f}"] = round(float((w | d).sum() * grid.res ** 2 / 1e6), 2)
    return out


In [24]:
# ================= PATCH 2: faster OSM loading + honest place names =================
import pandas as pd
# One Overpass query per layer (tiles only as a fallback): ~3 requests instead of ~60. Pre-event snapshot via [date:...].
def op_fast(bb, body, date, cols, geom):
    rows = _op_once(tuple(bb), body, date, cols, geom, tries=2)
    if rows is None:
        print("   single query failed -> falling back to tiles")
        return op_layer(make_tiles(bb, 0.06), body, date, cols, geom)
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
          gpd.GeoDataFrame({c: [] for c in cols + ["osmid"]}, geometry=[], crs=4326)
    return gdf.drop_duplicates("osmid") if len(gdf) else gdf

CJK = re.compile(r"[㐀-鿿]")
def tidy_places(pl):
    """Readable names; flag settlements whose OSM name is in Chinese script (likely across the border)."""
    pl = pl.copy()
    nm = pl["name"].fillna("").astype(str)
    en = pl["name:en"].fillna("").astype(str) if "name:en" in pl else pd.Series([""] * len(pl), index=pl.index)
    pl["foreign"] = nm.str.contains(CJK).values
    pl["name"] = [(e or f"{n} (Chinese-script name)") if f else (n or f"(unnamed {p})")
                  for n, e, f, p in zip(nm, en, pl["foreign"], pl["place"])]
    return pl

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    assert t < C["flood_date"], "OSM snapshot must be BEFORE the event (data rules)."
    pad = C.get("osm_pad_deg", 0.10)
    wide = (bb[0] - pad, bb[1] - pad, bb[2] + pad, bb[3] + pad)   # roads/places/hospitals: nearest town/hospital may be outside the AOI
    rc = ROAD_CLASSES.replace(",", "|")
    def key(*a): return ("osm2",) + a
    def layer(box_, body, cols, geom):
        return cached(key(box_, body, t), lambda: op_fast(box_, body, t, cols, geom))
    osm = dict(
        roads=layer(wide, f'way["highway"~"^({rc})$"]@@;', ["highway", "bridge", "name"], "line"),
        places=layer(wide, 'node["place"~"^(city|town|village|hamlet)$"]@@;', ["place", "name", "name:en"], "point"),
        hospitals=layer(wide, 'nwr["amenity"="hospital"]@@;nwr["healthcare"="hospital"]@@;', ["name"], "center"))
    if osm["hospitals"].empty:
        print("No hospitals found -> trying clinics")
        osm["hospitals"] = layer(wide, 'nwr["amenity"="clinic"]@@;', ["name"], "center")
    if len(osm["hospitals"]):                                       # node + way copies of the same hospital
        c = osm["hospitals"].geometry.centroid
        osm["hospitals"] = osm["hospitals"].loc[~pd.Series(list(zip(c.x.round(3), c.y.round(3)))).duplicated().values]
    osm["places"] = tidy_places(osm["places"]) if len(osm["places"]) else osm["places"].assign(foreign=[])
    # buildings: only near the detected mask (full-AOI building queries time out). This scope is reported in facts.
    fl = globals().get("_LAST_FLOOD")
    if fl is None or len(fl) == 0:
        print("No flood polygons -> no buildings loaded")
        osm["buildings"] = gpd.GeoDataFrame({"building": [], "name": []}, geometry=[], crs=4326)
    else:
        zone = shapely.union_all(fl.to_crs(4326).geometry.values).buffer(0.01)
        small = [tb for tb in make_tiles(bb, 0.03) if box(*tb).intersects(zone)][:60]
        print(f"buildings: {len(small)} tiles near the detected mask")
        osm["buildings"] = cached(key("bld", tuple(small), t),
                                  lambda: op_layer(small, 'way["building"]@@;', t, ["building", "name"], "poly"))
    for k, v in osm.items(): print(f"  {k}: {len(v)}")
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}


In [25]:
# ================= PATCH 3: facts that add up + one consistent bilingual report =================
import html as _html

def cutoff_analysis_v2(G_pre, G_post, osm, fz, snap_tol=500):
    pl = _cutoff_base(G_pre, G_post, osm, fz, snap_tol)
    # no town/hospital in OSM => say "not assessed", never "not connected"
    if len(pl) and not pl["town_status"].isin(["ok", "detour", "CUT_OFF"]).any():
        pl["town_status"] = "not_assessed"
    if len(pl) and len(osm["hospitals"]) == 0:
        pl["hospital_status"] = "not_assessed"
    return pl
if "_cutoff_base" not in globals():
    _cutoff_base = cutoff_analysis
cutoff_analysis = cutoff_analysis_v2

if "_bf_base2" not in globals():
    _bf_base2 = build_facts                       # keeps the earlier wrappers (cell 32)

def limitation_pairs(f):
    s, sens = f["settlements"], f.get("threshold_sensitivity_km2")
    L = [("Sentinel satellites revisit every few days; no early warning.",
          "Sentinel उपग्रहले केही दिनको अन्तरालमा मात्र पुनः तस्बिर लिन्छन्; पूर्व चेतावनी दिन सक्दैनन्।"),
         ("Overlap with the detected mask is not confirmed damage; nothing here is field-verified.",
          "पहिचान गरिएको क्षेत्रसँगको मिलान पुष्टि भएको क्षति होइन; क्षेत्रमा गएर कुनै जाँच गरिएको छैन।"),
         ("The detected area is a minimum: narrow or steep gorge flow paths and radar-shadow areas can be missed.",
          "पहिचान गरिएको क्षेत्र न्यूनतम अनुमान हो: साँघुरा वा ठाडा खोँच र रडार छायाँ क्षेत्रका बाढी-मार्ग छुट्न सक्छन्।"),
         ("OpenStreetMap is incomplete in remote valleys; roads and buildings missing from OSM cannot be analysed.",
          "दुर्गम उपत्यकामा OpenStreetMap अपूर्ण छ; OSM मा नभएका सडक र भवन विश्लेषण गर्न सकिँदैन।"),
         ("Cut-off status depends on a few OSM road segments; one flagged segment can isolate a whole branch.",
          "सम्पर्क टुटेको निष्कर्ष थोरै OSM सडक खण्डमा निर्भर छ; एउटै खण्डले पूरै शाखा अलग देखाउन सक्छ।"),
         ("Buildings were loaded only within about 1 km of the detected mask, so the building total is not a whole-area total.",
          "भवन पहिचान गरिएको क्षेत्रबाट करिब 1 किमी भित्र मात्र लोड गरिएको हो; त्यसैले भवन जम्मा सम्पूर्ण क्षेत्रको जम्मा होइन।")]
    if sens:
        ks = sorted(sens, key=float)
        txt_en = "; ".join(f"{sens[k]} km2 at {k} dB" for k in reversed(ks))
        txt_ne = "; ".join(f"{k} dB मा {sens[k]} वर्ग किमी" for k in reversed(ks))
        L.append((f"Detected area depends on the radar change threshold (radar only): {txt_en}.",
                  f"पहिचान गरिएको क्षेत्र रडार परिवर्तन सीमामा निर्भर छ (रडार मात्र): {txt_ne}।"))
    if isinstance(s.get("cut_off_from_town"), str):
        L.append(("No town was found in OSM for this area, so access to a town was not assessed.",
                  "यस क्षेत्रको OSM मा सहर भेटिएन; त्यसैले सहरसम्मको पहुँच मूल्याङ्कन गरिएको छैन।"))
    if s.get("cut_off_cross_border"):
        L.append(("Settlements with Chinese-script names are probably across the border; their road links to Nepali hospitals are not a Nepal-side result.",
                  "चिनियाँ लिपिमा नाम भएका बस्ती सम्भवतः सीमापारि छन्; नेपालका अस्पतालसँगको उनीहरूको सडक सम्बन्ध नेपाल-तर्फको नतिजा होइन।"))
    return L

def build_facts(C, info, stats, flood_area, places, path_table):
    f = _bf_base2(C, info, stats, flood_area, places, path_table)
    s = f["settlements"]; fo = places["foreign"] if "foreign" in places else False
    hs = places["hospital_status"]
    nm = lambda mask: sorted(places.loc[mask, "name"].dropna().tolist())
    s["cut_off_from_hospital"] = nm((hs == "CUT_OFF") & ~fo)
    s["cut_off_cross_border"] = nm((hs == "CUT_OFF") & fo)
    s["detour_to_hospital"] = nm(hs == "detour")
    s["inside_flood_zone"] = nm(places["in_flood"])
    s["analysed_for_hospital_access"] = int(hs.isin(["ok", "detour", "CUT_OFF"]).sum())
    s["no_road_data_within_500m"] = int((hs == "no_road_data").sum())
    s["not_connected_before_to_hospital"] = int((hs == "not_connected_before").sum())
    if (places["town_status"] == "not_assessed").all():
        s["cut_off_from_town"] = "not assessed: no town found in OSM data for this area"
    f["imagery"]["s1_gap_days"] = info.get("s1_gap_days")
    f["damage"]["buildings_scope"] = "OSM buildings within about 1 km of the detected mask"
    f["limitations"] = [en for en, _ in limitation_pairs(f)]
    return f

PT_NE = {"hamlet": "बस्ती", "village": "गाउँ", "town": "सहर", "city": "शहर"}
def full_report(f, lang="en"):
    """Deterministic report: every figure is copied from facts, EN and NE carry identical content."""
    en = lang == "en"
    im, d, s = f["imagery"], f["damage"], f["settlements"]
    water = "water" in f.get("flood_mask_kinds", [])
    cut, xb, det = s["cut_off_from_hospital"], s.get("cut_off_cross_border", []), s["detour_to_hospital"]
    town_na = isinstance(s["cut_off_from_town"], str)
    nc = s.get("not_connected_before_to_hospital", 0)
    S = []
    if en:
        S.append(("Summary", [f"Event date {f['event_date']}. Sentinel-1 {im['orbit']} pair on the same orbit track: before {im['s1_pre']}, "
                  f"after {im['s1_post']}" + (f" ({im['s1_gap_days']} days apart)" if im.get("s1_gap_days") else "") +
                  f". Sentinel-2 {'was used' if im['s2_used'] else 'was not used'} (clear-sky share {im['s2_clear_pct']}%). "
                  "All figures are satellite-derived estimates from an educational prototype."]))
        S.append(("Detected flooded / debris-affected area", [f"{f['flood_area_km2']} km2. " +
                  ("Includes open water and debris-affected areas. " if water else "Radar-change (debris-affected) areas only; no open water was detected. ") +
                  "This is a detected minimum, not a full extent."]))
        S.append(("Mapped assets that intersect the detected area (OpenStreetMap snapshot " + f["osm_snapshot"] + ")",
                  [f"Buildings: {d['buildings_hit']} intersect, {d['buildings_at_risk']} more within 30 m ({d['buildings_total']} OSM buildings checked, {d['buildings_scope'].replace('OSM buildings ', '')}).",
                   f"Roads: {d['road_km_hit']} km intersect (of {d['road_km_total']} km analysed).",
                   f"Bridges: {d['bridges_hit']} of {d['bridges_total']} intersect, {d['bridges_at_risk']} within 30 m."]))
        acc = [f"Settlements analysed for hospital access: {s['analysed_for_hospital_access']} of {s['total']}; no road within 500 m (not assessed): {s['no_road_data_within_500m']}; "
               f"already without a hospital route before the flood: {nc}.",
               "Cut off from the nearest hospital by the detected area: " + (", ".join(cut) if cut else "none detected") + "."]
        if xb: acc.append("Flagged but probably across the border (Chinese-script name): " + ", ".join(xb) + ".")
        if det: acc.append("Longer route only: " + ", ".join(det) + ".")
        acc.append("Access to a town: " + ("not assessed (no town found in OSM)." if town_na else (", ".join(s["cut_off_from_town"]) or "none cut off") + "."))
        S.append(("Access to hospitals", acc))
        if f["flood_path_settlements"]:
            S.append(("Settlements along the traced flood path", [f"- {p['name'] or '(unnamed)'} ({p['place']}): {p['km_downstream']} km downstream, {p['dist_to_path_m']} m from the path" for p in f["flood_path_settlements"]]))
        S.append(("Limitations", ["- " + en_ for en_, _ in limitation_pairs(f)]))
    else:
        S.append(("सारांश", [f"घटना मिति {f['event_date']}। Sentinel-1 ({im['orbit']} कक्षा, उही ट्र्याक) को अघिको {im['s1_pre']} र पछिको {im['s1_post']} का तस्बिर तुलना गरिएको" +
                  (f" ({im['s1_gap_days']} दिनको फरक)" if im.get("s1_gap_days") else "") +
                  f"। Sentinel-2 {'प्रयोग गरिएको छ' if im['s2_used'] else 'प्रयोग गरिएको छैन'} (खुला आकाश {im['s2_clear_pct']}%)। "
                  "सबै आँकडा उपग्रहबाट गरिएको अनुमान हुन्; यो शैक्षिक प्रोटोटाइप हो।"]))
        S.append(("पहिचान गरिएको बाढी / मलबा-प्रभावित क्षेत्र", [f"{f['flood_area_km2']} वर्ग किमी। " +
                  ("खुला पानी र मलबा-प्रभावित क्षेत्र दुवै समावेश छन्। " if water else "रडार परिवर्तन (मलबा-प्रभावित) क्षेत्र मात्र; खुला पानी भेटिएन। ") +
                  "यो पूर्ण क्षेत्र होइन, न्यूनतम पहिचान हो।"]))
        S.append(("पहिचान गरिएको क्षेत्रसँग जोडिएका संरचना (OpenStreetMap, " + f["osm_snapshot"] + ")",
                  [f"भवन: {d['buildings_hit']} क्षेत्रभित्र पर्छन्; थप {d['buildings_at_risk']} क्षेत्रबाट 30 मिटरको दूरीभित्र छन् ({d['buildings_total']} OSM भवन जाँचिए, पहिचान गरिएको क्षेत्रबाट करिब 1 किमी भित्रका)।",
                   f"सडक: {d['road_km_hit']} किमी क्षेत्रभित्र (विश्लेषण गरिएको {d['road_km_total']} किमी मध्ये)।",
                   f"पुल: {d['bridges_total']} मध्ये {d['bridges_hit']} क्षेत्रभित्र; क्षेत्रबाट 30 मिटरको दूरीभित्र: {d['bridges_at_risk']}।"]))
        acc = [f"अस्पताल पहुँच जाँचिएका बस्ती: {s['total']} मध्ये {s['analysed_for_hospital_access']}; 500 मिटर भित्र सडक नभएका (मूल्याङ्कन भएन): {s['no_road_data_within_500m']}; "
               f"बाढीअघि नै अस्पतालसँग सडक नजोडिएका: {nc}।",
               "पहिचान गरिएको क्षेत्रका कारण नजिकको अस्पतालसँग सडक सम्पर्क टुटेका बस्ती: " + (", ".join(cut) if cut else "कुनै पत्ता लागेन") + "।"]
        if xb: acc.append("चिन्ह लगाइएका तर सम्भवतः सीमापारि (चिनियाँ लिपिको नाम): " + ", ".join(xb) + "।")
        if det: acc.append("बाटो मात्र लामो भएका: " + ", ".join(det) + "।")
        acc.append("सहरसम्मको पहुँच: " + ("मूल्याङ्कन गरिएको छैन (OSM मा सहर भेटिएन)।" if town_na else (", ".join(s["cut_off_from_town"]) or "कुनै टुटेको छैन") + "।"))
        S.append(("अस्पतालसम्मको पहुँच", acc))
        if f["flood_path_settlements"]:
            S.append(("पत्ता लगाइएको बाढी-मार्गका बस्ती", [f"- {p['name'] or '(नाम नभएको)'} ({PT_NE.get(p['place'], p['place'])}): {p['km_downstream']} किमी तल्तिर, मार्गबाट {p['dist_to_path_m']} मिटर" for p in f["flood_path_settlements"]]))
        S.append(("सीमितताहरू", ["- " + ne_ for _, ne_ in limitation_pairs(f)]))
    return "\n".join(f"**{h}**\n" + "\n".join(lines) + "\n" for h, lines in S)

template_report = lambda f, lang="en": full_report(f, lang)   # also the fallback inside ask()

# ---- numbers_ok: ignore sensor names (Sentinel-1/2) but stay strict about everything else ----
def numbers_ok(text, facts):
    allowed = {float(x) for x in NUM.findall(json.dumps(facts, ensure_ascii=False))}
    txt = re.sub(r"Sentinel[\s‑-]*[12]", "Sentinel", text)
    txt = re.sub(r"(?<=\d),(?=\d{3})", "", txt)
    return [x for x in NUM.findall(txt) if float(x) not in allowed]

LLM_Q = ("Write a 3-4 sentence plain-language summary for rescue coordinators using only FACTS: the detected area, "
         "how many buildings/roads/bridges intersect it, and which settlements are cut off from a hospital (or that none were detected). "
         "Say it is an unverified satellite estimate. No lists, no headings, no markdown.")
def _plain(t):
    t = re.sub(r"[*_#`>]+", "", t)
    return " ".join(x.strip(" -•") for x in t.splitlines() if x.strip())

def situation_report(facts):
    """Base = deterministic facts report (always correct). AI summary = optional, validated against facts, else omitted."""
    out = {}
    for lang in ("en", "ne"):
        ai = llm(LLM_Q, facts, lang)
        out[lang] = dict(base=full_report(facts, lang), ai=_plain(ai) if ai else None)
        print(f"AI summary [{lang}]:", "validated" if ai else "FAILED number check -> omitted")
    return out

def _md(t):
    h, ul = [], False
    for ln in t.split("\n"):
        ln = ln.strip()
        if not ln:
            if ul: h.append("</ul>"); ul = False
            continue
        ln = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", _html.escape(ln))
        if ln.startswith("- "):
            if not ul: h.append("<ul>"); ul = True
            h.append(f"<li>{ln[2:]}</li>")
        else:
            if ul: h.append("</ul>"); ul = False
            h.append(f"<p>{ln}</p>")
    if ul: h.append("</ul>")
    return "\n".join(h)

def make_report_html(facts, texts, png_b64, path):
    d, im = facts["damage"], facts["imagery"]
    box_ = lambda lang, title, t: (f"<section lang='{lang}'><h3>{title}</h3>{_md(t['base'])}" +
        (f"<p class='ai'><b>{'AI summary (numbers checked against facts.json)' if lang == 'en' else 'AI सारांश (आँकडा facts.json सँग जाँचिएको)'}:</b> {_html.escape(t['ai'])}</p>" if t["ai"] else
         f"<p class='ai'><i>{'AI summary omitted: it failed the number check.' if lang == 'en' else 'AI सारांश राखिएको छैन: आँकडा जाँचमा असफल भयो।'}</i></p>") + "</section>")
    page = f"""<!doctype html><html><head><meta charset='utf-8'><title>Flood situation report {facts['event_date']}</title>
<style>body{{font:12px/1.45 'Noto Sans','Noto Sans Devanagari',sans-serif;max-width:820px;margin:auto;padding:0 10px}}
td,th{{border:1px solid #999;padding:3px 8px;text-align:center}}table{{border-collapse:collapse;margin:6px 0}}img{{max-width:100%}}
h3{{margin:14px 0 4px}}p{{margin:3px 0}}ul{{margin:2px 0 6px 18px;padding:0}}.ai{{background:#f3f6fa;padding:5px 8px;border-left:3px solid #4a76a8}}
.f{{font-size:9px;color:#555}}@media print{{body{{font-size:10px}}}}</style></head><body>
<h2>Flood situation report - {facts['event_date']}</h2>
<p class='f'>Imagery: Sentinel-1 {im['s1_pre']} (before) / {im['s1_post']} (after), {im['orbit']}. OpenStreetMap snapshot {facts['osm_snapshot']}. Educational prototype; estimates are not field-verified.</p>
<img alt='Detected flood / debris mask' src='data:image/png;base64,{png_b64}'>
<table><tr><th>Flooded / debris area</th><th>Buildings intersecting</th><th>Within 30 m</th><th>Road km intersecting</th><th>Bridges intersecting</th><th>Cut off from hospital</th></tr>
<tr><td>{facts['flood_area_km2']} km&sup2;</td><td>{d['buildings_hit']}</td><td>{d['buildings_at_risk']}</td><td>{d['road_km_hit']}</td><td>{d['bridges_hit']}</td><td>{len(facts['settlements']['cut_off_from_hospital'])}{(' (+' + str(len(facts['settlements'].get('cut_off_cross_border', []))) + ' cross-border)') if facts['settlements'].get('cut_off_cross_border') else ''}</td></tr></table>
{box_('en', 'English', texts['en'])}{box_('ne', 'नेपाली', texts['ne'])}
<p class='f'>{ATTR}<br>Educational prototype; not an operational tool.</p></body></html>"""
    open(path, "w", encoding="utf-8").write(page)


In [26]:
# ================= PATCH 4: dashboard that a rescue coordinator can read in 10 seconds =================
STATUS = {"CUT_OFF": ("red", "Road link to hospital LOST"), "detour": ("orange", "Longer route to hospital"),
          "ok": ("green", "Route to hospital kept"), "no_road_data": ("gray", "No road data nearby - not assessed"),
          "not_connected_before": ("black", "No hospital route even before the flood"), "not_assessed": ("gray", "Not assessed")}

def make_dashboard(grid, C, flood, b, r, places, hosp, flood_line, path, facts=None):
    cx, cy = grid.to_ll.transform((grid.bounds[0] + grid.bounds[2]) / 2, (grid.bounds[1] + grid.bounds[3]) / 2)
    m = folium.Map(location=[cy, cx], zoom_start=11, tiles=None)
    folium.TileLayer("https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png", name="Basemap: light",
                     attr="&copy; OpenStreetMap contributors &copy; CARTO", show=True).add_to(m)
    w, s_, e, n = C["bbox"]
    folium.Rectangle([[s_, w], [n, e]], color="#555", weight=1, dash_array="4", fill=False, tooltip="Analysed area").add_to(m)
    color = {"water": "#1f78b4", "debris": "#8c5a2b"}; label = {"water": "Open water (new)", "debris": "Radar-change / debris-affected area"}
    for kind in ("water", "debris"):
        g = flood[flood.kind == kind]
        if len(g):
            folium.GeoJson(ll(g).to_json(), name=f"Flood mask: {label[kind]}", tooltip=label[kind],
                           style_function=lambda _, c=color[kind]: dict(color=c, fillColor=c, fillOpacity=.55, weight=1)).add_to(m)
    bh = b[b.hit | b.at_risk]
    if len(bh):
        folium.GeoJson(ll(bh[["hit", "at_risk", "geometry"]]).to_json(), name="Buildings: intersect mask (red) / within 30 m (orange)",
                       style_function=lambda f: dict(color="red" if f["properties"]["hit"] else "orange", weight=1, fillOpacity=.7)).add_to(m)
    rh = r[r.hit]
    if len(rh):
        folium.GeoJson(ll(rh[["highway", "is_bridge", "geometry"]]).to_json(), name="Roads (red) / bridges (purple) intersecting mask",
                       style_function=lambda f: dict(color="purple" if f["properties"]["is_bridge"] else "red", weight=5)).add_to(m)
    fg = folium.FeatureGroup(name="Settlements (colour = hospital access)").add_to(m)
    for _, p in ll(places).iterrows():
        col, txt = STATUS.get(p["hospital_status"], ("gray", p["hospital_status"]))
        if p.get("foreign"): txt += " (name in Chinese script: probably across the border)"
        a, z = p.get("hospital_km_pre"), p.get("hospital_km_post")
        route = f"<br>Route to hospital: {a} km before, {z} km after" if a is not None and z is not None and a == a and z == z else ""
        folium.CircleMarker([p.geometry.y, p.geometry.x], radius=8 if p["hospital_status"] == "CUT_OFF" else 6, color=col, fill=True,
                            fill_opacity=.9, tooltip=f"<b>{p['name']}</b> ({p['place']})<br>{txt}{route}").add_to(fg)
    fh = folium.FeatureGroup(name="Hospitals").add_to(m)
    for _, h in ll(hosp).iterrows():
        pt = h.geometry.centroid
        folium.Marker([pt.y, pt.x], icon=folium.Icon(color="black", icon="plus"), tooltip=h.get("name") or "hospital (unnamed)").add_to(fh)
    if flood_line is not None:
        folium.GeoJson(gpd.GeoSeries([flood_line], crs=grid.epsg).to_crs(4326).to_json(), name="Traced flood path (DEM)",
                       style_function=lambda _: dict(color="cyan", weight=3, dashArray="6")).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    panel = ""
    if facts:
        d, im, s = facts["damage"], facts["imagery"], facts["settlements"]
        cut = ", ".join(s["cut_off_from_hospital"]) or "none detected"
        panel = (f"<details open><summary><b>Flood {facts['event_date']}</b> - satellite estimate, not field-verified</summary>"
                 f"<table style='font-size:12px'><tr><td><b>{facts['flood_area_km2']} km&sup2;</b></td><td>detected area (a minimum)</td></tr>"
                 f"<tr><td><b>{d['buildings_hit']}</b> (+{d['buildings_at_risk']})</td><td>buildings in mask (+ within 30 m)</td></tr>"
                 f"<tr><td><b>{d['road_km_hit']} km</b></td><td>road in mask (of {d['road_km_total']} km)</td></tr>"
                 f"<tr><td><b>{d['bridges_hit']}</b> of {d['bridges_total']}</td><td>bridges in mask</td></tr>"
                 f"<tr><td><b>{len(s['cut_off_from_hospital'])}</b></td><td>cut off from hospital: {cut}</td></tr>"
                 f"<tr><td><b>{s['analysed_for_hospital_access']}</b> / {s['total']}</td><td>settlements assessable ({s['no_road_data_within_500m']} have no road data)</td></tr></table>"
                 f"<div style='color:#555'>Sentinel-1 {im['orbit']} {im['s1_pre']} &rarr; {im['s1_post']}; Sentinel-2 {'used' if im['s2_used'] else 'not used'} ({im['s2_clear_pct']}% clear)</div>"
                 f"</details>")
    legend = "".join(f"<div><span style='display:inline-block;width:10px;height:10px;border-radius:5px;background:{c};margin-right:6px'></span>{t}</div>"
                     for k, (c, t) in STATUS.items() if k != "not_assessed")
    m.get_root().html.add_child(folium.Element(
        f"<div style='position:fixed;bottom:36px;left:10px;z-index:9998;background:#fffe;font:12px sans-serif;padding:8px 10px;border-radius:6px;"
        f"max-width:310px;box-shadow:0 1px 4px #0005'>{panel}<div style='margin-top:6px'>{legend}</div></div>"
        f"<div style='position:fixed;bottom:0;left:0;right:0;z-index:9999;background:#fffc;font:11px sans-serif;padding:3px'>"
        f"{ATTR} Educational prototype - not an operational tool.</div>"))
    m.save(path)
    return m


def static_png(grid, cls, perm, dem=None, roads=None, places=None, flood=None):
    """Report map: hillshade + roads + detected area (outlined, enlarged so tiny areas are visible) + cut-off settlements."""
    minx, miny, maxx, maxy = grid.bounds
    fig, ax = plt.subplots(figsize=(6.4, 6.4 * (maxy - miny) / (maxx - minx)))
    if dem is not None:
        gy, gx = np.gradient(ndi.gaussian_filter(dem.astype("float64"), 1), grid.res)
        hs = np.clip(0.5 + 0.5 * (-gx * 0.7 - gy * 0.7) / 2.0, 0, 1)            # simple NW-lit hillshade
        ax.imshow(hs, cmap="gray", extent=(minx, maxx, miny, maxy), alpha=.55, vmin=0, vmax=1)
    rgb = np.zeros(cls.shape + (4,)); rgb[perm] = (.2, .45, .85, .5)
    rgb[cls == 1] = (.12, .47, .71, 1); rgb[cls == 2] = (.55, .35, .17, 1)
    ax.imshow(rgb, extent=(minx, maxx, miny, maxy))
    if roads is not None and len(roads): roads.plot(ax=ax, color="#555", linewidth=.5)
    if flood is not None and len(flood):
        flood.buffer(150).boundary.plot(ax=ax, color="#c0392b", linewidth=1.2, linestyle="--")
    if places is not None and len(places):
        c = places[places["hospital_status"] == "CUT_OFF"]
        if len(c): ax.scatter(c.geometry.x, c.geometry.y, s=40, c="red", edgecolor="k", zorder=5)
    ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy); ax.axis("off")
    ax.set_title("Detected flood / debris area (dashed red ring = enlarged outline)\nbrown = radar-change, blue = new water, light blue = permanent water, red dot = cut off from hospital", fontsize=8)
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


In [27]:
# ================= PATCH 5: run() with everything wired together + consistency check =================
def run(C=CONFIG):
    assert C["osm_date"] < C["flood_date"], "OSM snapshot must be BEFORE the event (data rules)."
    grid = Grid(C["bbox"], C["res"]); print(f"Grid {grid.W}x{grid.H} @ {grid.res:.0f} m, EPSG:{grid.epsg}")
    F = map_flood(grid, C)
    if F["info"]["s1_valid_pct"] < 50:
        raise RuntimeError(f"Sentinel-1 coverage too low ({F['info']['s1_valid_pct']}%).")
    flood = vectorize(F["cls"], grid)
    flood_area = round(float((F["cls"] > 0).sum() * grid.res ** 2 / 1e6), 2)
    osm = load_osm(grid, C)
    b, r, fz, stats = damage_assessment(osm, flood)
    G0 = build_graph(r); G1 = remove_flooded_edges(G0, fz)
    places = cutoff_analysis(G0, G1, osm, fz)
    line, ptable = None, None
    if C.get("upstream_point"):
        try: line, ptable = trace_flood_path(F["dem"], grid, C["upstream_point"], osm["places"])
        except Exception as ex: print("Flood-path step skipped:", ex)
    facts = build_facts(C, F["info"], stats, flood_area, places, ptable)
    facts["threshold_sensitivity_km2"] = sensitivity(F, grid, C)
    facts["limitations"] = [en for en, _ in limitation_pairs(facts)]
    texts = situation_report(facts)
    out = C["out_dir"]
    m = make_dashboard(grid, C, flood, b, r, places, osm["hospitals"], line, f"{out}/dashboard.html", facts)
    make_report_html(facts, texts, static_png(grid, F["cls"], F["perm_water"], F["dem"], r, places, flood), f"{out}/situation_report.html")
    json.dump(facts, open(f"{out}/facts.json", "w"), ensure_ascii=False, indent=1)
    flood.to_crs(4326).to_file(f"{out}/flood.geojson", driver="GeoJSON")
    print(json.dumps(facts["damage"], indent=1)); print(texts["en"]["base"])
    return dict(grid=grid, flood=flood, facts=facts, places=places, osm=osm, map=m, F=F)

def consistency_check(RESULT, out_dir=None):
    """Run after run(): every output file must tell the same story. Prints PASS/FAIL and the numbers for README/slides."""
    out_dir = out_dir or CONFIG["out_dir"]
    f = json.load(open(f"{out_dir}/facts.json"))
    rep = open(f"{out_dir}/situation_report.html", encoding="utf-8").read()
    dash = open(f"{out_dir}/dashboard.html", encoding="utf-8").read()
    g = gpd.read_file(f"{out_dir}/flood.geojson").to_crs(RESULT["grid"].epsg)
    s, d, im = f["settlements"], f["damage"], f["imagery"]
    poly_km2 = float(g.area.sum()) / 1e6
    pdays = (dt.date.fromisoformat(im["s1_post"]) - dt.date.fromisoformat(im["s1_pre"])).days
    checks = [
        ("polygon area ~ facts area (+-15% or 0.01 km2)", abs(poly_km2 - f["flood_area_km2"]) <= max(0.15 * f["flood_area_km2"], 0.01)),
        ("settlement counts add up to total", s["analysed_for_hospital_access"] + s["no_road_data_within_500m"] + s.get("not_connected_before_to_hospital", 0) == s["total"]),
        ("OSM snapshot is before the event", f["osm_snapshot"] < f["event_date"]),
        ("S1 pair is a multiple of 12 days on one orbit", pdays % 12 == 0),
        ("report has no debug/fallback text", "failed number validation" not in rep and "**" not in rep),
        ("report shows area / buildings / bridges from facts", all(str(x) in rep for x in (f["flood_area_km2"], d["buildings_hit"], d["bridges_hit"]))),
        ("dashboard shows the same area and building count", str(f["flood_area_km2"]) in dash and str(d["buildings_hit"]) in dash),
        ("every cut-off name is in the dashboard", all(n in dash for n in s["cut_off_from_hospital"])),
        ("no API key pattern in outputs", not re.search(r"gsk_[A-Za-z0-9]{20}|sh-[0-9a-f]{8}-", rep + dash)),
    ]
    for name, ok in checks: print(("PASS  " if ok else "FAIL  ") + name)
    print("\nNumbers to paste into README / slides / report (single source of truth = facts.json):")
    print(f"  area {f['flood_area_km2']} km2 | buildings {d['buildings_hit']} (+{d['buildings_at_risk']}) of {d['buildings_total']} | "
          f"road {d['road_km_hit']} of {d['road_km_total']} km | bridges {d['bridges_hit']} of {d['bridges_total']}")
    print(f"  settlements {s['total']}: analysed {s['analysed_for_hospital_access']}, no road data {s['no_road_data_within_500m']}, "
          f"unconnected before {s.get('not_connected_before_to_hospital', 0)} | cut off: {s['cut_off_from_hospital'] or 'none'} | cross-border: {s.get('cut_off_cross_border') or 'none'}")
    print(f"  S1 {im['s1_pre']} -> {im['s1_post']} ({pdays} d, {im['orbit']}) | S2 used: {im['s2_used']} ({im['s2_clear_pct']}% clear) | sensitivity {f.get('threshold_sensitivity_km2')}")
    return all(ok for _, ok in checks)


In [29]:
def process(data, evalscript, grid, tries=5):
    body = {"input": {"bounds": {"bbox": list(grid.bounds),
                                 "properties": {"crs": f"http://www.opengis.net/def/crs/EPSG/0/{grid.epsg}"}},
                      "data": data},
            "output": {"width": grid.W, "height": grid.H,
                       "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
            "evalscript": evalscript}
    last = None
    for a in range(tries):
        try:
            r = requests.post(PROCESS_URL, json=body, headers=auth_headers(), timeout=(30, 600))
            if r.status_code == 200:
                with MemoryFile(r.content) as mf, mf.open() as ds:
                    return ds.read().astype("float32")
            last = f"{r.status_code}: {r.text[:300]}"
            if r.status_code not in (429, 500, 502, 503, 504):
                break                                   # a real error such as 400/401: retrying won't help
        except (requests.ConnectionError, requests.Timeout) as ex:
            last = str(ex)[:200]
        wait = 15 * (a + 1)
        print(f"   Process API retry {a+1}/{tries} in {wait}s ({last[:80]})")
        time.sleep(wait)
    raise RuntimeError(f"Process API failed after {tries} tries: {last}")

In [40]:
if "_fr0" not in globals(): _fr0 = full_report
def full_report(f, lang="en"):
    t = _fr0(f, lang)
    if isinstance(f["settlements"]["cut_off_from_town"], list):
        t = t.replace("Access to a town: ", "Cut off from the nearest town: ").replace("सहरसम्मको पहुँच: ", "नजिकको सहरसँग सडक सम्पर्क टुटेका: ")
    return t

In [41]:
RESULT = run()

Grid 1169x1568 @ 30 m, EPSG:32645
S1 pair: {'pre_day': '2026-08-16', 'post_day': '2026-08-28', 'orbit': 'ASCENDING', 'rel': 85}


/tmp/ipykernel_937/1225178076.py:41: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  c = osm["hospitals"].geometry.centroid


buildings: 60 tiles near the detected mask
  roads: 5144
  places: 380
  hospitals: 18
  buildings: 39477
AI summary [en]: validated
AI summary [ne]: validated
{
 "buildings_total": 39477,
 "buildings_hit": 1796,
 "buildings_at_risk": 820,
 "road_km_total": 4155.4,
 "road_km_hit": 35.51,
 "bridges_total": 149,
 "bridges_hit": 15,
 "bridges_at_risk": 3,
 "buildings_scope": "OSM buildings within about 1 km of the detected mask"
}
**Summary**
Event date 2026-08-26. Sentinel-1 ASCENDING pair on the same orbit track: before 2026-08-16, after 2026-08-28 (12 days apart). Sentinel-2 was used (clear-sky share 18.9%). All figures are satellite-derived estimates from an educational prototype.

**Detected flooded / debris-affected area**
9.77 km2. Includes open water and debris-affected areas. This is a detected minimum, not a full extent.

**Mapped assets that intersect the detected area (OpenStreetMap snapshot 2026-07-27)**
Buildings: 1796 intersect, 820 more within 30 m (39477 OSM buildings che

## 12. Open the dashboard, chat with the copilot, evaluate

In [42]:
from IPython.display import display
display(RESULT["map"])   # interactive dashboard (also saved as outputs/dashboard.html)

In [43]:
print(ask("Which villages cannot reach a hospital by road, and how many buildings were hit?", RESULT["facts"], "en"))
print(ask("कुन बस्तीहरू अस्पताल पुग्न सक्दैनन्?", RESULT["facts"], "ne"))

**Villages that cannot reach a hospital by road**  
Resuo (गाउँ)

**Buildings hit**  
1796 buildings

*These results are satellite‑derived estimates from an educational prototype.*

**Limitations**  
- Sentinel satellites revisit every few days; no early warning.  
- Overlap with the detected mask is not confirmed damage; nothing here is field‑verified.  
- The detected area is a minimum: narrow or steep gorge flow paths and radar‑shadow areas can be missed.  
- OpenStreetMap is incomplete in remote valleys; roads and buildings missing from OSM cannot be analysed.  
- Cut‑off status depends on a few OSM road segments; one flagged segment can isolate a whole branch.  
- Buildings were loaded only within about 1 km of the detected mask, so the building total is not a whole‑area total.  
- Detected area depends on the radar change threshold (radar only): 1.56 km² at 4.0 dB; 4.59 km² at 3.0 dB; 29.13 km² at 2.0 dB.  
- Settlements with Chinese‑script names are probably across the border; t

### Consistency check (run after `run()`): all outputs must tell the same story

In [44]:
ALL_OK = consistency_check(RESULT)
assert ALL_OK, 'Fix the FAIL lines above before submitting.'

PASS  polygon area ~ facts area (+-15% or 0.01 km2)
PASS  settlement counts add up to total
PASS  OSM snapshot is before the event
PASS  S1 pair is a multiple of 12 days on one orbit
PASS  report has no debug/fallback text
PASS  report shows area / buildings / bridges from facts
PASS  dashboard shows the same area and building count
PASS  every cut-off name is in the dashboard
PASS  no API key pattern in outputs

Numbers to paste into README / slides / report (single source of truth = facts.json):
  area 9.77 km2 | buildings 1796 (+820) of 39477 | road 35.51 of 4155.4 km | bridges 15 of 149
  settlements 380: analysed 243, no road data 96, unconnected before 41 | cut off: ['(unnamed hamlet)', '(unnamed hamlet)', '(unnamed hamlet)', 'Somdang', 'आमाछोदिङ्मो', 'गोगने', 'गोठेन', 'गोदाम', 'ग्रे', 'जर्सा डाँडा', 'ठुलो भार्खु', 'तातोपानी', 'तेटाङचेट', 'नौविसे फाँट', 'पाजुङ', 'बाहुनडाँडा', 'बेत्रावती', 'ब्रिमडाङ', 'भञ्ज्याङ', 'भैंसे', 'भ्रजम', 'मैलुङ', 'रसुवागढी', 'लिङलिङ', 'सानाे भार्खु', 'सि

In [45]:
import zipfile, glob, pandas as pd, geopandas as gpd
for z in glob.glob("/content/EMSR927_AOI0*.zip"):
    zipfile.ZipFile(z).extractall("/content/emsr927/" + z.split("/")[-1][:-4])

def load(pat):
    fs = sorted(glob.glob(f"/content/emsr927/**/*{pat}*.shp", recursive=True))
    print(pat, "->", [f.split("/")[-1] for f in fs])
    return pd.concat([gpd.read_file(f) for f in fs], ignore_index=True)

aoi, ev = load("areaOfInterestA"), load("observedEventA")     # AOI polygons + mapped event extent
ev.to_file("/content/emsr927_extent.geojson", driver="GeoJSON")

flood_in_aoi = gpd.clip(RESULT["flood"], aoi.to_crs(RESULT["flood"].crs))   # score only where the reference looked
print("reference extent km2: %.2f" % (ev.to_crs(32645).area.sum() / 1e6))
print("your mask inside the AOIs km2: %.3f" % (flood_in_aoi.to_crs(32645).area.sum() / 1e6))
print(evaluate_against_reference(flood_in_aoi, "/content/emsr927_extent.geojson", RESULT["grid"]))

areaOfInterestA -> ['EMSR927_AOI01_GRA_PRODUCT_areaOfInterestA_v1.shp', 'EMSR927_AOI02_GRA_PRODUCT_areaOfInterestA_v2.shp']
observedEventA -> ['EMSR927_AOI01_GRA_PRODUCT_observedEventA_v1.shp', 'EMSR927_AOI02_GRA_PRODUCT_observedEventA_v2.shp']
reference extent km2: 2.40
your mask inside the AOIs km2: 0.420
{'IoU': 0.16, 'precision': 0.929, 'recall': 0.162, 'F1': 0.276}


In [46]:
b, r, fz, st = damage_assessment(RESULT["osm"], RESULT["flood"])
pts = load("builtUpP").to_crs(b.crs)
dam = pts[pts.damage_gra.isin(["Destroyed", "Damaged"])]
b_in = gpd.clip(b, aoi.to_crs(b.crs))
print("OSM buildings inside AOIs:", len(b_in), "| flagged as hit:", int(b_in.hit.sum()))
print("reference destroyed/damaged buildings:", len(dam),
      "| possibly damaged:", int((pts.damage_gra == "Possibly damaged").sum()))
print("reference destroyed bridges: 6 | yours flagged:", st["bridges_hit"])

builtUpP -> ['EMSR927_AOI01_GRA_PRODUCT_builtUpP_v1.shp', 'EMSR927_AOI02_GRA_PRODUCT_builtUpP_v2.shp']
OSM buildings inside AOIs: 0 | flagged as hit: 0
reference destroyed/damaged buildings: 760 | possibly damaged: 104
reference destroyed bridges: 6 | yours flagged: 15


### Coverage / tolerance diagnostics (checking only - do not tune on them)

In [47]:
# ---- Extra diagnostics for the EMSR927 comparison. CHECKING ONLY: do not tune thresholds or the bbox on these numbers. ----
from shapely.geometry import box as _box
crs_ = RESULT["grid"].epsg
aoi_p = aoi.to_crs(crs_); ref_p = ev.to_crs(crs_); pred = RESULT["flood"]
bbox_p = gpd.GeoSeries([_box(*CONFIG["bbox"])], crs=4326).to_crs(crs_).iloc[0]
cover = aoi_p.geometry.intersection(bbox_p).area.sum() / aoi_p.geometry.area.sum()
print(f"coverage : {cover:.0%} of the EMSR927 AOI area lies inside our analysed bbox"
      "  (low value = a coverage problem, not a detection problem)")
ref_u = shapely.union_all(ref_p.geometry.values)
pred_u = shapely.union_all(pred.geometry.values) if len(pred) else shapely.Polygon()
for tol in (0, 90):                                    # 90 m = 3 pixels: DEM / orthorectification can shift features
    near = ref_u.intersection(pred_u.buffer(tol)).area / ref_u.area if tol else ref_u.intersection(pred_u).area / ref_u.area
    print(f"recall   : {near:.1%} of the reference area is within {tol} m of a detection")
print(f"size     : detected {pred_u.area/1e6:.3f} km2 vs reference {ref_u.area/1e6:.2f} km2 "
      f"(ratio {pred_u.area/ref_u.area:.3f})")
print("polygons : detected", len(pred), "| touching an AOI:", int(pred.intersects(shapely.union_all(aoi_p.geometry.values)).sum()))


coverage : 100% of the EMSR927 AOI area lies inside our analysed bbox  (low value = a coverage problem, not a detection problem)
recall   : 16.1% of the reference area is within 0 m of a detection
recall   : 50.0% of the reference area is within 90 m of a detection
size     : detected 9.756 km2 vs reference 2.40 km2 (ratio 4.057)
polygons : detected 881 | touching an AOI: 35


In [48]:
# Download everything (dashboard.html, situation_report.html, facts.json, flood.geojson)
import shutil
shutil.make_archive("floodwatch_outputs", "zip", CONFIG["out_dir"])
try:
    from google.colab import files; files.download("floodwatch_outputs.zip")
except Exception: pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>